# Cell 1: Environment Setup

In [ ]:
import os
import math
import copy
import time
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, balanced_accuracy_score, roc_auc_score, confusion_matrix
)


# ==============================================================================
# v12 — PART 1: the LACA lag-bias g(Δτ) is restored to the original mlp
# Linear(1,32)-Tanh-Linear(32,1) (97 params) as a single permanent implementation
# (rbf / bucketed and the form-selection scaffolding are removed). PART 2: a
# SEQUENTIAL Mamba fusion-backbone sweep — num_blocks, then d_state, then d_conv —
# with everything else FIXED. SEED is the single source for all seeding here.
# ==============================================================================
SEED = 42
LACA_BETA = 1.0   # LACA temporal_bias_scale — FIXED (v9 sweep winner)
PIPELINE_CONFIG = {
    "Audio bin selection": "strongest-64 (N=64 frequency bins, v6 winner; TRAINING audio only, frozen for val/test)",
    "Audio time positions": "Ta = 64 per window (FIXED, v7)",
    "Sensor input":        "raw 6 IMU channels (accel x/y/z, gyro x/y/z), Ts = 128 (FIXED; no derived magnitudes)",
    "LACA beta":           f"temporal_bias_scale = {LACA_BETA} (FIXED, v9 sweep winner)",
    "LACA g(Δτ)":          "mlp Linear(1,32)-Tanh-Linear(32,1), 97 params (FIXED; v10 form + capacity comparisons)",
    "LACA heads":          "num_heads = 4 (FIXED, conservative default kept after v11)",
    "PART 2 SWEEP":        "Mamba: A num_blocks {2,4,6} -> B d_state {8,16,32} -> C d_conv {2,4,6} "
                           "(sequential; default 4 / 16 / 4)",
    "Audio windowing":     "none (raw 1536-sample chunk -> rfft; no Hann)",
    "Audio compression":   "sqrt(magnitude)",
    "Audio normalization": "none",
    "Classification loss": "class-weighted cross-entropy (inverse TRAIN-set class frequency)",
    "Seed":                f"{SEED} (random / numpy / torch / cuda, cuDNN deterministic; reset for every condition)",
    "Everything else":     "v10 unchanged (sensor path, split, 20 epochs, AdamW, eval)",
}
print("=" * 90)
print("  AEROSYNC-MAMBA v12 — RESTORED BASELINE + MAMBA CONFIGURATION SWEEP")
print("=" * 90)
for _k, _v in PIPELINE_CONFIG.items():
    print(f"  • {_k:<20}: {_v}")
print("=" * 90)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

# ==============================================================================
# FIX 2: install the REAL mamba_ssm package (selective-scan SSM kernels).
#
# STRATEGY (in order):
#   1. PREBUILT WHEEL from the causal-conv1d / mamba-ssm GitHub releases, matched
#      to this exact Python / torch / CUDA / C++ ABI build. This is the ONLY
#      reliable path on Kaggle — building from source routinely hangs or OOMs
#      the kernel because Kaggle's toolchain doesn't line up with what these
#      packages' setup.py expects.
#   2. Capped from-source build (240s timeout, output captured) as a fallback
#      if no matching prebuilt wheel exists for this environment.
#   3. Pure-PyTorch fallback block (Cell 2) if both of the above fail — this is
#      the safety net that guarantees the notebook can NEVER hang/crash here,
#      but every mode in the frequency-bin sweep shares whichever backbone this
#      cell resolves to, so the 6-way comparison stays apples-to-apples either way.
# ==============================================================================
MAMBA_INSTALL_TIMEOUT_S = 240

def _detect_env_tags():
    """Python tag, torch major.minor, CUDA major.minor, C++11 ABI flag — the
    axes prebuilt causal-conv1d / mamba-ssm wheels are keyed on."""
    import sys
    py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    torch_mm = ".".join(torch.__version__.split("+")[0].split(".")[:2])
    cuda_ver = torch.version.cuda  # e.g. "12.1", or None on CPU-only torch
    try:
        abi_flag = bool(torch._C._GLIBCXX_USE_CXX11_ABI)
    except Exception:
        abi_flag = bool(getattr(torch, "compiled_with_cxx11_abi", lambda: False)())
    return py_tag, torch_mm, cuda_ver, abi_flag

def _github_releases(repo, timeout_s=20):
    import urllib.request, json as _json
    url = f"https://api.github.com/repos/{repo}/releases"
    req = urllib.request.Request(url, headers={"User-Agent": "kaggle-notebook", "Accept": "application/vnd.github+json"})
    with urllib.request.urlopen(req, timeout=timeout_s) as resp:
        return _json.load(resp)

def _find_prebuilt_wheel_url(repo, py_tag, torch_mm, cuda_ver, abi_flag):
    """Best-matching .whl asset URL from `repo`'s GitHub releases, or None."""
    try:
        releases = _github_releases(repo)
    except Exception as e:
        print(f"    could not query github.com/{repo}/releases: {e!r}")
        return None

    abi_str = f"cxx11abi{'TRUE' if abi_flag else 'FALSE'}"
    cuda_major = cuda_ver.split(".")[0] if cuda_ver else None
    cuda_tag_exact = f"cu{cuda_ver.replace('.', '')}" if cuda_ver else None   # e.g. "cu121"
    exact, major_only = [], []

    for rel in releases:
        for asset in rel.get("assets", []):
            name = asset.get("name", "")
            if not name.endswith(".whl"):
                continue
            if py_tag not in name or abi_str not in name or f"torch{torch_mm}" not in name:
                continue
            if cuda_tag_exact and cuda_tag_exact in name:
                exact.append(asset["browser_download_url"])
            elif cuda_major and f"cu{cuda_major}" in name:
                major_only.append(asset["browser_download_url"])

    chosen = (exact or major_only)
    return chosen[0] if chosen else None

def _pip_run(args, timeout_s=MAMBA_INSTALL_TIMEOUT_S):
    """pip invocation that can NEVER hang or crash the kernel: hard timeout,
    output captured (not streamed — a runaway nvcc/ninja build can print
    megabytes), every exception caught."""
    import subprocess, sys
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + args
    try:
        proc = subprocess.run(cmd, timeout=timeout_s, capture_output=True, text=True, check=False)
        if proc.returncode != 0:
            tail = (proc.stderr or proc.stdout or "")[-500:]
            print(f"    ⚠️  pip install {args} failed (exit {proc.returncode}); tail:\n{tail}")
            return False
        return True
    except subprocess.TimeoutExpired:
        print(f"    ⚠️  pip install {args} exceeded {timeout_s}s — killed to avoid hanging/OOM-ing "
              f"the kernel; treating as failed.")
        return False
    except Exception as e:
        print(f"    ⚠️  pip install {args} raised {e!r}; treating as failed.")
        return False

def _install_via_prebuilt_wheel(pip_name, repo, py_tag, torch_mm, cuda_ver, abi_flag):
    url = _find_prebuilt_wheel_url(repo, py_tag, torch_mm, cuda_ver, abi_flag)
    if url is None:
        print(f"    no matching prebuilt wheel found for {pip_name} "
              f"(py={py_tag}, torch={torch_mm}, cuda={cuda_ver}, cxx11abi={abi_flag})")
        return False
    print(f"    found prebuilt wheel: {url}")
    return _pip_run([url, "--no-deps"])

_MAMBA_INSTALL_METHOD = "not attempted (no CUDA GPU)"

if torch.cuda.is_available():
    try:
        import mamba_ssm  # noqa: F401
        print("mamba_ssm already installed.")
        _MAMBA_INSTALL_METHOD = "already installed"
    except ImportError:
        py_tag, torch_mm, cuda_ver, abi_flag = _detect_env_tags()
        print(f"Environment: python={py_tag} | torch={torch_mm} | cuda={cuda_ver} | cxx11abi={abi_flag}")
        print("Installing real mamba_ssm + causal-conv1d (selective-scan SSM kernels)...")

        print("  [1/3] trying prebuilt wheels (no compilation — the reliable path on Kaggle)...")
        ok1 = _install_via_prebuilt_wheel("causal-conv1d", "Dao-AILab/causal-conv1d", py_tag, torch_mm, cuda_ver, abi_flag)
        ok2 = _install_via_prebuilt_wheel("mamba-ssm", "state-spaces/mamba", py_tag, torch_mm, cuda_ver, abi_flag) if ok1 else False

        if ok1 and ok2:
            _MAMBA_INSTALL_METHOD = "prebuilt wheel"
        else:
            print(f"  [2/3] no complete prebuilt-wheel match — falling back to a capped "
                  f"from-source build ({MAMBA_INSTALL_TIMEOUT_S}s timeout per package)...")
            # --no-build-isolation avoids re-resolving/reinstalling torch inside an isolated
            # build env, a common cause of causal-conv1d's build spinning forever on Kaggle.
            ok1b = _pip_run(["causal-conv1d>=1.2.0", "--no-build-isolation"])
            ok2b = _pip_run(["mamba-ssm", "--no-build-isolation"]) if ok1b else False
            _MAMBA_INSTALL_METHOD = "source build" if (ok1b and ok2b) else "failed"

        print(f"  [3/3] resolution: {_MAMBA_INSTALL_METHOD}")
        if _MAMBA_INSTALL_METHOD in ("prebuilt wheel", "source build"):
            print(f"✅ causal-conv1d + mamba-ssm installed via {_MAMBA_INSTALL_METHOD}.")
        else:
            print("⚠️  mamba_ssm install did not complete — the pure-PyTorch FALLBACK block will be "
                  "used (see Cell 2 / Cell 6 output). This does not affect the frequency-bin sweep: "
                  "every mode uses the same backbone, so the 6-way comparison is still apples-to-apples.")
else:
    print("⚠️  No CUDA GPU detected — mamba_ssm requires CUDA, so the pure-PyTorch FALLBACK block will be used (see Cell 2 / Cell 6 output).")


# ==============================================================================
# GLOBAL SEEDING — python `random`, numpy, torch CPU + every CUDA device, plus
# deterministic cuDNN. The mission split in Cell 4 keeps its own
# random.seed(42); this covers everything else (weight init, dropout, DataLoader
# shuffle order). set_all_seeds() is called again right before model
# construction for every condition of the Mamba configuration sweep (Cell 6).
# Note: mamba_ssm's CUDA selective-scan backward uses atomic adds, so GPU runs
# can still differ in the last few decimal places even with identical seeds.
# ==============================================================================
def set_all_seeds(seed: int = SEED) -> int:
    """Reset python / numpy / torch (CPU + all CUDA devices) RNGs to `seed`."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    return seed

set_all_seeds(SEED)
print(f"✅ Seeds set -> random={SEED}, numpy={SEED}, torch={SEED}, cuda={SEED} (cuDNN deterministic)")

# Cell 2: Bi-Modal AeroSync-Mamba Architecture (HTT + LACA + Mamba) — mlp g(Δτ) restored (97 params), configurable Mamba backbone

In [ ]:
# --- Fusion backbone resolution: REAL mamba_ssm (CUDA selective-scan) vs. pure-PyTorch FALLBACK ---
_MAMBA_CLASS = None
_FUSION_BACKBONE_REASON = None
try:
    if torch.cuda.is_available():
        from mamba_ssm import Mamba as _MAMBA_CLASS
        _FUSION_BACKBONE_REASON = "mamba_ssm.Mamba (REAL selective-scan SSM: hidden state h_t, learned A/B/C, discretized Δ) — CUDA GPU detected"
    else:
        _FUSION_BACKBONE_REASON = "FALLBACK (VectorizedSSMBlock, pure PyTorch, NO SSM recurrence) — no CUDA GPU detected; mamba_ssm requires CUDA"
except ImportError as _mamba_import_err:
    _FUSION_BACKBONE_REASON = f"FALLBACK (VectorizedSSMBlock, pure PyTorch, NO SSM recurrence) — mamba_ssm import failed: {_mamba_import_err}"

print(f"\n{'='*70}\nFUSION BACKBONE RESOLUTION: {_FUSION_BACKBONE_REASON}\n{'='*70}\n")


# --- 1. Continuous Temporal Positional Embedding & HTT ---
class ContinuousTimeEmbedding(nn.Module):
    def __init__(self, d_model: int, num_frequencies: int = 64):
        super().__init__()
        self.frequencies = nn.Parameter(
            torch.exp(torch.linspace(0, math.log(1000.0), num_frequencies)), requires_grad=False
        )
        self.mlp = nn.Sequential(
            nn.Linear(2 * num_frequencies, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

    def forward(self, tau: torch.Tensor) -> torch.Tensor:
        if tau.dim() == 2:
            tau = tau.unsqueeze(-1)
        angles = tau * self.frequencies * 2.0 * math.pi
        fourier = torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)
        return self.mlp(fourier)


class HeterogeneousTemporalTokenization(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        self.d_model = d_model
        # 0: Audio, 1: Sensor Telemetry
        self.modality_embeddings = nn.Embedding(2, d_model)
        self.time_embedding = ContinuousTimeEmbedding(d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, z_bar: torch.Tensor, tau: torch.Tensor, modality_idx: int) -> torch.Tensor:
        mod_idx = torch.tensor(modality_idx, device=z_bar.device, dtype=torch.long)
        e_m = self.modality_embeddings(mod_idx).view(1, 1, self.d_model)
        phi_tau = self.time_embedding(tau)
        return self.norm(z_bar + e_m + phi_tau)


# --- 2. Bi-Modal LACA (Audio <-> Telemetry Alignment) ---
class PairwiseLACA(nn.Module):
    def __init__(self, d_model: int, num_heads: int = 4, beta: float = 1.0):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.beta = beta

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.lag_bias = nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 1))

    def forward(self, H_m, H_n, tau_m, tau_n):
        B, T_m, _ = H_m.shape
        _, T_n, _ = H_n.shape
        Q = self.w_q(H_m).view(B, T_m, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.w_k(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.w_v(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        delta_tau = (tau_m.unsqueeze(2) - tau_n.unsqueeze(1)).unsqueeze(-1)
        B_mn = self.lag_bias(delta_tau).squeeze(-1).unsqueeze(1)

        attn_weights = F.softmax(scores + self.beta * B_mn, dim=-1)
        out = torch.matmul(attn_weights, V).transpose(1, 2).contiguous().view(B, T_m, self.d_model)
        H_tilde_mn = self.out_proj(out)

        A_mn = attn_weights.mean(dim=1)
        tau_diff = tau_n.unsqueeze(1) - tau_m.unsqueeze(2)
        expected_lag = (A_mn * tau_diff).sum(dim=(1, 2)) / float(T_m)
        return H_tilde_mn, A_mn, expected_lag


class BiModalLACAFusion(nn.Module):
    def __init__(self, d_model: int, num_heads: int = 4, beta: float = 1.0):
        super().__init__()
        self.laca_as = PairwiseLACA(d_model, num_heads, beta)
        self.laca_sa = PairwiseLACA(d_model, num_heads, beta)
        self.gamma_as = nn.Parameter(torch.tensor(0.5))
        self.gamma_sa = nn.Parameter(torch.tensor(0.5))

    def forward(self, Ha, Hs, tau_a, tau_s):
        H_a_from_s, _, lag_as = self.laca_as(Ha, Hs, tau_a, tau_s)
        H_s_from_a, _, lag_sa = self.laca_sa(Hs, Ha, tau_s, tau_a)

        H_tilde_a = Ha + self.gamma_as * H_a_from_s
        H_tilde_s = Hs + self.gamma_sa * H_s_from_a
        return H_tilde_a, H_tilde_s, {"lag_as": lag_as, "lag_sa": lag_sa}


# --- 3a. FALLBACK: pure-PyTorch gated-conv block (NOT a real SSM) ---
# Used ONLY when mamba_ssm / CUDA is unavailable (see resolution printed
# above). This block has NO hidden state h_t, NO A/B/C matrices, and NO
# discretization step — it is a depthwise-conv + gated-MLP block, kept only
# as a last-resort fallback so the notebook still runs on CPU-only machines.
class VectorizedSSMBlock(nn.Module):
    def __init__(self, d_model: int, d_conv: int = 4, expand: int = 2):
        super().__init__()
        self.d_inner = int(expand * d_model)
        self.in_proj = nn.Linear(d_model, self.d_inner * 2, bias=False)
        self.conv1d = nn.Conv1d(
            self.d_inner, self.d_inner, kernel_size=d_conv,
            padding=d_conv - 1, groups=self.d_inner, bias=True
        )
        self.mixer = nn.Sequential(
            nn.Linear(self.d_inner, self.d_inner),
            nn.GELU(),
            nn.Linear(self.d_inner, self.d_inner)
        )
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, D = x.shape
        residual = x
        x_norm = self.norm(x)
        xz = self.in_proj(x_norm)
        x_branch, z_branch = xz.chunk(2, dim=-1)

        x_conv = self.conv1d(x_branch.transpose(1, 2))[:, :, :L].transpose(1, 2)
        x_conv = F.silu(x_conv)

        y = self.mixer(x_conv) * F.silu(z_branch)
        return residual + self.out_proj(y)


# --- 3b. REAL Mamba block (genuine selective-scan SSM) ---
# Thin wrapper around mamba_ssm.Mamba, which implements the actual S6
# recurrence (hidden state h_t, learned continuous A/B/C matrices, and a
# discretization step Δ), wrapped in the same pre-norm + residual pattern as
# the fallback block above so it is a drop-in replacement.
#
# v12 — local-conv implementation. mamba_ssm's fused path runs the depthwise
# causal conv through the causal-conv1d CUDA kernel, which only supports widths
# 2..4 (csrc/causal_conv1d.cpp: TORCH_CHECK(width >= 2 && width <= 4)). For any
# other d_conv the block uses mamba_ssm's own non-fused path with the plain
# nn.Conv1d causal conv (same parameters, same math; the selective scan is still
# the real CUDA kernel). "fused" is used whenever the kernel supports the width,
# so the v5-v10 default (d_conv=4) runs exactly as before.
import sys
import contextlib

CAUSAL_CONV1D_WIDTHS = (2, 3, 4)
DEFAULT_MAMBA_CONFIG = (4, 16, 4)   # (num_mamba_blocks, d_state, d_conv) — v5-v10 values


def _fused_conv_available():
    if _MAMBA_CLASS is None:
        return False
    return getattr(sys.modules[_MAMBA_CLASS.__module__], "causal_conv1d_fn", None) is not None


def resolve_conv_impl(d_conv: int, force_pytorch: bool = False) -> str:
    if _MAMBA_CLASS is None:
        return "native"            # fallback VectorizedSSMBlock: plain nn.Conv1d, any width
    if not _fused_conv_available():
        return "pytorch"           # causal-conv1d not importable -> mamba_ssm already uses nn.Conv1d
    if force_pytorch or d_conv not in CAUSAL_CONV1D_WIDTHS:
        return "pytorch-forced"    # kernel present but bypassed for this block
    return "fused"


@contextlib.contextmanager
def _causal_conv1d_kernel_disabled():
    """Temporarily hide causal_conv1d_fn from mamba_ssm so Mamba.forward takes its
    `self.conv1d(x)[..., :seqlen]` branch. Scoped to one block's forward call."""
    mod = sys.modules[_MAMBA_CLASS.__module__]
    assert hasattr(mod, "causal_conv1d_fn"), (
        f"{mod.__name__} has no causal_conv1d_fn global — this mamba_ssm version cannot be switched to the "
        "PyTorch conv path safely")
    saved = mod.causal_conv1d_fn
    mod.causal_conv1d_fn = None
    try:
        yield
    finally:
        mod.causal_conv1d_fn = saved


class RealMambaBlock(nn.Module):
    def __init__(self, d_model: int, d_conv: int = 4, expand: int = 2, d_state: int = 16,
                 force_pytorch_conv: bool = False):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mamba = _MAMBA_CLASS(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
        self.conv_impl = resolve_conv_impl(d_conv, force_pytorch_conv)
        if self.conv_impl == "pytorch-forced":
            self.mamba.use_fast_path = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.conv_impl == "pytorch-forced":
            with _causal_conv1d_kernel_disabled():
                return x + self.mamba(self.norm(x))
        return x + self.mamba(self.norm(x))


def build_fusion_blocks(d_model, num_blocks, d_state, d_conv, force_pytorch_conv=False, expand=2):
    if _MAMBA_CLASS is not None:
        return nn.ModuleList([
            RealMambaBlock(d_model=d_model, d_conv=d_conv, expand=expand, d_state=d_state,
                           force_pytorch_conv=force_pytorch_conv) for _ in range(num_blocks)
        ])
    # the fallback block has no SSM state, so d_state does not apply to it
    return nn.ModuleList([
        VectorizedSSMBlock(d_model=d_model, d_conv=d_conv, expand=expand) for _ in range(num_blocks)
    ])


# --- 4. Full Bi-Modal AeroSync-Mamba Model ---
class BiModalAeroSyncMamba(nn.Module):
    def __init__(
        self,
        d_model: int = 256,
        n_classes: int = 5,
        num_mamba_blocks: int = 4,
        dropout: float = 0.15,
        temporal_bias_scale: float = 1.0,
        audio_in_dim: int = 64,
        sensor_in_dim: int = 6,
        mamba_d_state: int = 16,
        mamba_d_conv: int = 4,
        mamba_force_pytorch_conv: bool = False,
    ):
        super().__init__()
        self.d_model = d_model
        self.n_classes = n_classes

        # Sensor 1D-CNN encoder
        self.sensor_encoder = nn.Sequential(
            nn.Conv1d(sensor_in_dim, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64, d_model, kernel_size=3, padding=1), nn.BatchNorm1d(d_model), nn.GELU(),
        )
        # Audio feature projector
        self.audio_encoder = nn.Sequential(
            nn.Linear(audio_in_dim, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Linear(d_model, d_model)
        )

        self.htt = HeterogeneousTemporalTokenization(d_model)
        self.laca = BiModalLACAFusion(d_model=d_model, num_heads=4, beta=temporal_bias_scale)
        self.sep_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        # --- FIX 2: use the REAL mamba_ssm block when available, otherwise the
        # pure-PyTorch fallback (see resolution printed at the top of this cell,
        # and the unmissable print in Cell 6 right after model construction). ---
        # v12: (num_mamba_blocks, d_state, d_conv) are configurable. The default
        # (4, 16, 4) is built exactly as in v5-v10. Any other configuration first
        # builds and DISCARDS the default stack so the global RNG advances exactly as
        # in v5-v10, then builds the real stack from the SAME starting RNG state and
        # restores the post-default state. Result: every non-Mamba module (built
        # before AND after this point) gets bit-identical initial weights in every
        # condition, and the post-construction RNG stream (shuffle order, dropout
        # masks) is identical too — only the fusion backbone differs.
        self.mamba_config = {"num_blocks": num_mamba_blocks, "d_state": mamba_d_state, "d_conv": mamba_d_conv}
        if (num_mamba_blocks, mamba_d_state, mamba_d_conv) == DEFAULT_MAMBA_CONFIG:
            self.mamba_blocks = build_fusion_blocks(d_model, num_mamba_blocks, mamba_d_state, mamba_d_conv,
                                                    mamba_force_pytorch_conv)
        else:
            _rng_pre = torch.get_rng_state()
            _rng_placeholder = build_fusion_blocks(d_model, *DEFAULT_MAMBA_CONFIG)
            del _rng_placeholder
            _rng_post = torch.get_rng_state()
            torch.set_rng_state(_rng_pre)
            self.mamba_blocks = build_fusion_blocks(d_model, num_mamba_blocks, mamba_d_state, mamba_d_conv,
                                                    mamba_force_pytorch_conv)
            torch.set_rng_state(_rng_post)
        self.fusion_backbone_reason = _FUSION_BACKBONE_REASON

        self.pool_attn = nn.Sequential(nn.Linear(d_model, 64), nn.Tanh(), nn.Linear(64, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, n_classes),
        )

        self.aux_a = nn.Linear(d_model, n_classes)
        self.aux_s = nn.Linear(d_model, n_classes)

    def forward(self, xa, xs, ta, ts):
        B = xa.shape[0]

        # 1. Modality Encoders
        za = self.audio_encoder(xa)
        zs = self.sensor_encoder(xs.transpose(1, 2)).transpose(1, 2)

        # 2. HTT (0: Audio, 1: Sensor)
        Ha = self.htt(za, ta, modality_idx=0)
        Hs = self.htt(zs, ts, modality_idx=1)

        # 3. Bi-Modal LACA Alignment
        H_tilde_a, H_tilde_s, aux = self.laca(Ha, Hs, ta, ts)
        za_p = H_tilde_a.mean(dim=1)
        zs_p = H_tilde_s.mean(dim=1)

        # 4. Cross-Modal Mamba Fusion: [ H_tilde_a ; [SEP] ; H_tilde_s ]
        sep = self.sep_token.expand(B, -1, -1)
        H_fused = torch.cat([H_tilde_a, sep, H_tilde_s], dim=1)

        F_seq = H_fused
        for blk in self.mamba_blocks:
            F_seq = blk(F_seq)

        weights = F.softmax(self.pool_attn(F_seq), dim=1)
        zf = (F_seq * weights).sum(dim=1)
        logits = self.classifier(zf)

        pooled_reps = {"z_a": za_p, "z_s": zs_p}
        aux_preds = {"y_a": self.aux_a(za_p), "y_s": self.aux_s(zs_p)}
        return logits, pooled_reps, aux_preds, aux

print("Bi-Modal AeroSync-Mamba model class defined successfully!")

# Cell 3: Bi-Modal Loss Function (Classification + Audio-IMU InfoNCE + KL)

In [ ]:
class BiModalAeroSyncLoss(nn.Module):
    def __init__(self, lambda_align=0.1, lambda_cons=0.1, temperature=0.07, class_weights=None):
        super().__init__()
        self.lambda_align = lambda_align
        self.lambda_cons = lambda_cons
        self.temperature = temperature
        # --- FIX 4: class-weighted classification loss. `class_weights` is an
        # inverse-frequency tensor computed from REAL training-set class counts
        # in Cell 6 (printed there for verification), not hardcoded here. ---
        self.ce = nn.CrossEntropyLoss(weight=class_weights)

    def _infonce(self, z1, z2):
        z1_norm = F.normalize(z1, p=2, dim=-1)
        z2_norm = F.normalize(z2, p=2, dim=-1)
        sim = torch.matmul(z1_norm, z2_norm.t()) / self.temperature
        labels = torch.arange(z1.shape[0], device=z1.device)
        return 0.5 * (F.cross_entropy(sim, labels) + F.cross_entropy(sim.t(), labels))

    def forward(self, logits, targets, pooled, aux_preds):
        # 1. Fault Classification Loss
        loss_cls = self.ce(logits, targets)

        # 2. Audio-Sensor Contrastive Alignment Loss
        loss_align = self._infonce(pooled["z_a"], pooled["z_s"])

        # 3. Auxiliary Consistency Loss
        p_fused = F.softmax(logits.detach(), dim=-1)
        loss_cons = (
            F.kl_div(F.log_softmax(aux_preds["y_a"], dim=-1), p_fused, reduction="batchmean") +
            F.kl_div(F.log_softmax(aux_preds["y_s"], dim=-1), p_fused, reduction="batchmean")
        ) / 2.0

        total = loss_cls + self.lambda_align * loss_align + self.lambda_cons * loss_cons
        return total, {"loss_cls": float(loss_cls.item()), "loss_align": float(loss_align.item())}

def compute_all_metrics(y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    try:
        auroc = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
    except Exception:
        auroc = float("nan")
    return {"accuracy": acc, "macro_f1": f1, "balanced_accuracy": bal_acc, "macro_auroc": auroc}

print("Loss & Metrics helper ready!")

# Cell 4 (Self-Downloading & Preprocessing): Real TII UAV Dataset

In [ ]:
import os
import json
import random
import numpy as np
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm
import torch

DATASET_BASE = Path("/kaggle/working/tii_uav_dataset")
DATASET_ROOT = DATASET_BASE / "Dataset"
CACHE_DIR = Path("/kaggle/working/preprocessed_bimodal_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# 1. Automatically clone the official repository if not present
if not DATASET_ROOT.exists():
    print("📥 Cloning official TII UAV Realistic Fault Dataset from GitHub...")
    !git clone --depth 1 https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git /kaggle/working/tii_uav_dataset
    print("✅ Dataset cloned successfully!")

_chunk_duration_log = []
_fft_shape_printed = False  # verification: print the FFT feature shape once

# Three different sizes — kept as separate named constants so they are never confused:
N_AUDIO_BINS = 64             # audio FREQUENCY axis: strongest-64 bins = feature dim per time step (FIXED, v6 winner)
TA_AUDIO = 64                 # audio TIME axis: time positions per window (FIXED, v7)
TS_SENSOR = 128               # sensor TIME axis: time positions per window (FIXED, unchanged from v6)
_audio_chunks_per_window = []  # verification: real audio chunks available in each window (vs Ta)
_MISSION_CACHE = {}  # mission_dir -> parse_real_mission(...) output, parsed ONCE and reused for every N

# 2. Real Flight Mission Parser (6-Axis IMU + Audio)
def parse_real_mission(mission_dir: Path):
    sensor_matches = list(mission_dir.glob("*SensorCombined*.jsonl"))
    audio_matches  = list(mission_dir.glob("*AudioBuffer*.jsonl"))
    if not sensor_matches or not audio_matches:
        return None

    # 2.1 Parse Real IMU
    sensor_times, accel_data, gyro_data = [], [], []
    with open(sensor_matches[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            try:
                rec = json.loads(line)
                t_sec = float(rec["timestamp"]) / 1e6
                sensor_times.append(t_sec)
                accel_data.append(rec.get("accelerometer_m_s2", [0.0, 0.0, 9.81]))
                gyro_data.append(rec.get("gyro_rad", [0.0, 0.0, 0.0]))
            except Exception:
                continue

    if len(sensor_times) < 50: return None
    sensor_times = np.array(sensor_times, dtype=np.float64)
    sensor_times = sensor_times - sensor_times[0]
    telemetry = np.concatenate([np.array(accel_data, dtype=np.float32), np.array(gyro_data, dtype=np.float32)], axis=-1)

    # 2.2 Parse Real Audio
    audio_chunks = []
    with open(audio_matches[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            try:
                rec = json.loads(line)
                samples = rec.get("audio_data", rec.get("buffer", rec.get("data", [])))
                if len(samples) > 0: audio_chunks.append(samples)
            except Exception:
                continue

    if len(audio_chunks) < 10: return None
    audio_chunks = np.array(audio_chunks, dtype=np.float32)

    # --- FIX: derive REAL per-chunk audio timestamps ---
    # `recording_started_time` is a constant repeated on every line (not a
    # real per-chunk timestamp), and the previous fixed 0.10s/chunk guess
    # doesn't match the data (empirically ~17-19ms/chunk). Instead, derive
    # chunk duration from this mission's REAL, IMU-verified flight duration,
    # since audio and IMU are recorded within the same continuous rosbag
    # session.
    mission_duration_s = sensor_times[-1] - sensor_times[0]  # already zero-based
    n_chunks = len(audio_chunks)
    chunk_duration_s = mission_duration_s / n_chunks
    audio_times = np.arange(n_chunks, dtype=np.float64) * chunk_duration_s

    global _chunk_duration_log
    _chunk_duration_log.append(chunk_duration_s)

    return {
        "sensor_times": sensor_times,
        "telemetry": telemetry,
        "audio_times": audio_times,
        "audio_chunks": audio_chunks,
    }


def get_mission(mission_dir: Path):
    """parse_real_mission, memoized: every mission's JSONL is parsed once and the
    same arrays are reused for every N (identical to re-parsing each time)."""
    if mission_dir not in _MISSION_CACHE:
        _MISSION_CACHE[mission_dir] = parse_real_mission(mission_dir)
    return _MISSION_CACHE[mission_dir]


# 2.3 Strongest-N FFT bin selection (TRAINING-SET audio only, unsupervised, leakage-safe)
def select_strongest_audio_bins(train_mission_dirs, n_bins):
    """
    Selects the `n_bins` FFT magnitude bins with the highest mean magnitude,
    computed ONLY from training-set audio (raw waveform energy — never class
    labels). Called once per N, before any train/val/test window slicing for
    that N, so the exact same frozen indices are reused identically across all
    three splits.

    Identical criterion to the v5 strongest-64 selection; only the budget N is
    a parameter now. Returns (sorted bin indices, data-derived bin width in Hz
    or None).
    """
    print(f"[STRONGEST-{n_bins}] Computing bin-selection statistics from TRAINING-SET audio only "
          f"({len(train_mission_dirs)} training missions) — BEFORE any train/validation/test "
          "window preprocessing.")

    bin_magnitude_sum = None
    bin_count = 0
    fft_len_ref = None
    local_chunk_durations = []

    for m_dir in tqdm(train_mission_dirs, desc=f"Bin selection N={n_bins} (train-only audio scan)", leave=False):
        mission_data = get_mission(m_dir)
        if mission_data is None:
            continue
        audio = mission_data["audio_chunks"]
        a_times = mission_data["audio_times"]
        if len(a_times) > 1:
            local_chunk_durations.append((a_times[-1] - a_times[0]) / (len(a_times) - 1))

        mag = np.abs(np.fft.rfft(audio, axis=1))  # (n_chunks, fft_len), unsupervised energy
        if fft_len_ref is None:
            fft_len_ref = mag.shape[1]
        min_len = min(fft_len_ref, mag.shape[1])
        mag = mag[:, :min_len]
        if bin_magnitude_sum is not None and len(bin_magnitude_sum) > min_len:
            bin_magnitude_sum = bin_magnitude_sum[:min_len]
        fft_len_ref = min_len

        chunk_sum = mag.sum(axis=0)
        bin_magnitude_sum = chunk_sum if bin_magnitude_sum is None else bin_magnitude_sum + chunk_sum
        bin_count += mag.shape[0]

    assert bin_magnitude_sum is not None and bin_count > 0, (
        f"STRONGEST-{n_bins} FAILED: no training-set audio chunks were available for bin selection."
    )

    mean_magnitude = bin_magnitude_sum / bin_count  # unsupervised magnitude/energy criterion
    top_bins = np.sort(np.argsort(mean_magnitude)[::-1][:n_bins]).astype(int)

    assert len(top_bins) == n_bins, (
        f"STRONGEST-{n_bins} FAILED: selected {len(top_bins)} bins, expected {n_bins}"
    )

    # --- Hz labeling ---
    # This notebook does not document a raw microphone sample rate anywhere,
    # so we estimate an effective sample rate from quantities the notebook
    # already derives from real data: samples-per-audio-buffer (inverted from
    # the rFFT length) and the real per-mission inter-buffer timing computed
    # above (same derivation `parse_real_mission` already uses for
    # `chunk_duration_s`). This is a data-derived estimate used only for
    # human-readable labeling below — it never affects which bins are
    # selected or how features are built.
    samples_per_buffer = fft_len_ref * 2 - 2
    est_sample_rate_hz = (
        samples_per_buffer / np.mean(local_chunk_durations) if local_chunk_durations else None
    )
    bin_width_hz = est_sample_rate_hz / samples_per_buffer if est_sample_rate_hz else None

    print(f"[STRONGEST-{n_bins}] Selected {n_bins} bins from {bin_count} training-set audio chunks "
          f"across {len(train_mission_dirs)} missions (FFT length = {fft_len_ref}).")
    print(f"[STRONGEST-{n_bins}] Selected bin indices (ascending): {top_bins.tolist()}")
    if bin_width_hz:
        print(f"[STRONGEST-{n_bins}] Estimated audio sample rate ≈ {est_sample_rate_hz:.1f} Hz "
              "(derived from audio buffer length and real inter-buffer timing across training "
              "missions; the dataset does not document a mic sample rate directly, so this is "
              "a data-derived estimate for labeling only, not a hardcoded spec).")
        # contiguous runs of selected bins -> approx. frequency bands
        runs, start = [], top_bins[0]
        for prev, cur in zip(top_bins[:-1], top_bins[1:]):
            if cur != prev + 1:
                runs.append((start, prev))
                start = cur
        runs.append((start, top_bins[-1]))
        print(f"[STRONGEST-{n_bins}] Selected bins as {len(runs)} contiguous band(s) -> approx. frequency (Hz):")
        for lo, hi in runs:
            print(f"    bins {lo:3d}-{hi:3d} ({hi - lo + 1:3d} bins)  ->  {lo * bin_width_hz:8.1f} - {hi * bin_width_hz:8.1f} Hz")
    else:
        print(f"[STRONGEST-{n_bins}] Could not estimate a sample rate (insufficient timing data); "
              "printing bin indices only.")

    return top_bins, bin_width_hz


# 2.4 Audio feature extraction — the single, permanent path (v5), width = N:
#     raw chunk -> rfft (no window) -> |.| -> frozen strongest-N bins -> sqrt
#     (no normalization).
def compute_audio_features(audio_chunks, bin_indices):
    """
    audio_chunks : (n_chunks, chunk_len) raw waveform chunks of one mission
    bin_indices  : frozen strongest-N bin indices from select_strongest_audio_bins
                   (computed once per N from TRAINING audio; never recomputed here)
    returns      : (n_chunks, N) float32 = sqrt(|rfft|) at those bins, N = len(bin_indices)
    """
    assert bin_indices is not None and len(bin_indices) > 0, (
        "FEATURES FAILED: strongest-N bin indices must be computed from training audio "
        "(select_strongest_audio_bins) before any feature extraction."
    )
    n_bins = len(bin_indices)

    audio_fft_mag = np.abs(np.fft.rfft(audio_chunks, axis=1))  # (n_chunks, chunk_len//2 + 1), REAL FFT
    n_chunks_mission = audio_fft_mag.shape[0]
    n_bins_available = audio_fft_mag.shape[1]
    valid_idx = bin_indices[bin_indices < n_bins_available]

    a_feats = np.zeros((n_chunks_mission, n_bins), dtype=np.float32)
    a_feats[:, :len(valid_idx)] = np.sqrt(audio_fft_mag[:, valid_idx]).astype(np.float32)

    assert a_feats.shape == (n_chunks_mission, n_bins), (
        f"FEATURES FAILED: audio feature array shape is {a_feats.shape}, "
        f"expected (n_chunks={n_chunks_mission}, N={n_bins})"
    )
    return a_feats


# 3. Slicing into 2.56s Diagnostic Samples
def slice_real_bimodal(mission_data, class_label, bin_indices, Ta, window_sec=2.56, step_sec=1.28, Ts=128):
    # Ta = audio TIME positions per window (required, no default); Ts = sensor TIME positions (unchanged).
    # The frequency width of every audio time step is n_bins = len(bin_indices), independent of Ta.
    global _fft_shape_printed
    s_times = mission_data["sensor_times"]
    telemetry = mission_data["telemetry"]
    a_times = mission_data["audio_times"]
    audio = mission_data["audio_chunks"]

    assert bin_indices is not None and len(bin_indices) > 0, (
        "SLICE FAILED: strongest-N bin indices must be computed from training audio "
        "(select_strongest_audio_bins) before any window preprocessing."
    )

    # --- strongest-N audio features: FFT magnitude spectrum, frozen top-N bins ---
    # Previously: the first 64 FFT bins (low frequencies only) were used
    # unconditionally for every split. Now: the 64 bins with the highest mean
    # magnitude in TRAINING-set audio (computed once in
    # select_strongest_audio_bins, unsupervised, leakage-safe) are frozen and
    # reused identically for train, validation, and test.
    # FFT (no window) -> frozen strongest-N bins -> sqrt, via compute_audio_features (2.4).
    n_bins = len(bin_indices)
    a_feats = compute_audio_features(audio, bin_indices)
    n_chunks_mission = audio.shape[0]

    assert a_feats.shape == (n_chunks_mission, n_bins), (
        f"STRONGEST-{n_bins} FAILED: audio feature array shape is {a_feats.shape}, "
        f"expected (n_chunks={n_chunks_mission}, {n_bins})"
    )

    if not _fft_shape_printed:
        print(f"[STRONGEST-{n_bins} VERIFICATION] audio chunk shape (raw waveform): {audio.shape} "
              f"| np.fft.rfft output shape: {(audio.shape[0], audio.shape[1] // 2 + 1)} "
              f"| final a_feats (sqrt-compressed, frozen strongest-{n_bins} bins) shape: {a_feats.shape}")
        print(f"[STRONGEST-{n_bins} VERIFICATION] Confirmed: features were produced via np.fft.rfft "
              f"(frequency-domain magnitude spectrum) using the frozen strongest-{n_bins} bin indices, "
              f"NOT the first {n_bins} bins.")
        _fft_shape_printed = True

    max_time = min(s_times[-1], a_times[-1])
    samples = []
    t_start = 0.0

    while t_start + window_sec <= max_time:
        t_end = t_start + window_sec
        s_mask = (s_times >= t_start) & (s_times <= t_end)
        if s_mask.sum() < 10:
            t_start += step_sec
            continue

        # --- FIX 1: REAL per-window sensor timestamps ---
        # Previously: target_s_t = np.linspace(t_start, t_end, Ts) — a synthetic
        # uniform grid that is IDENTICAL for every window (window_sec is fixed),
        # so nothing real ever reached the model downstream. Now: pick Ts
        # evenly-spaced INDICES into the actual raw IMU sample timestamps that
        # fall inside this window. Real IMU logging is not perfectly uniform, so
        # these timestamps genuinely vary window-to-window and mission-to-mission.
        s_idx_in_window = np.where(s_mask)[0]
        real_s_pos = np.linspace(0, len(s_idx_in_window) - 1, Ts).round().astype(int)
        real_s_idx = s_idx_in_window[real_s_pos]
        target_s_t = s_times[real_s_idx]  # REAL raw sensor timestamps (seconds, mission-relative)

        s_window = np.zeros((Ts, 6), dtype=np.float32)
        for c in range(6):
            s_window[:, c] = np.interp(target_s_t, s_times[s_mask], telemetry[s_mask, c])

        # --- FIX 1: REAL per-window audio timestamps ---
        # audio_times itself is a per-mission uniform grid derived from the
        # mission's real chunk_duration_s (see parse_real_mission above), which
        # genuinely varies mission-to-mission (see the chunk-duration min/max
        # printed at the end of this cell). We pick real indices from it within
        # this window rather than an independent synthetic linspace.
        a_mask = (a_times >= t_start) & (a_times <= t_end)
        _audio_chunks_per_window.append(int(a_mask.sum()))
        if a_mask.sum() >= 2:
            a_idx_in_window = np.where(a_mask)[0]
            real_a_pos = np.linspace(0, len(a_idx_in_window) - 1, Ta).round().astype(int)
            real_a_idx = a_idx_in_window[real_a_pos]
            target_a_t = a_times[real_a_idx]  # REAL per-mission audio timestamps
        else:
            # Fallback only for the rare window with too few real audio samples
            target_a_t = np.linspace(t_start, t_end, Ta)

        a_window = np.zeros((Ta, n_bins), dtype=np.float32)  # (TIME positions, FREQUENCY bins)
        for c in range(n_bins):
            a_window[:, c] = np.interp(target_a_t, a_times, a_feats[:, c], left=0.0, right=0.0)

        assert a_window.shape == (Ta, n_bins), (
            f"STRONGEST-{n_bins} FAILED: window audio feature shape is {a_window.shape}, "
            f"expected ({Ta}, {n_bins})"
        )

        samples.append({
            "x_a": a_window,
            "x_s": s_window,
            "label": int(class_label),
            "tau_a": (target_a_t - t_start).astype(np.float32),  # REAL, zero-based, seconds
            "tau_s": (target_s_t - t_start).astype(np.float32),  # REAL, zero-based, seconds
            "window_sec": np.float32(window_sec),
        })
        t_start += step_sec

    return samples

# --- VERIFICATION: no leftover mode-switching in the audio feature path ---
import inspect
_feat_params = list(inspect.signature(compute_audio_features).parameters)
_slice_params = list(inspect.signature(slice_real_bimodal).parameters)
_switch_names = ("window_fn", "compression", "norm_stats", "normalization", "audio_mode", "mode")
_switch_params = [p for p in _feat_params + _slice_params if p in _switch_names]
try:
    _feat_src = inspect.getsource(compute_audio_features)
    _dead_branches = [t for t in ("hanning", "log1p", "log10", "zscore", "window_fn", "compression ==",
                                  "norm_stats", "elif") if t in _feat_src]
except (OSError, TypeError):
    _feat_src, _dead_branches = None, []
_stale_globals = [n for n in ("AUDIO_WINDOW_FNS", "AUDIO_COMPRESSIONS", "LOG_DB_EPS", "get_audio_window",
                              "compute_train_norm_stats", "N_BUDGETS", "TA_VALUES", "SENSOR_CHANNEL_CONFIGS",
                              "DERIVED_CHANNEL_SOURCES") if n in globals()]

assert _feat_params == ["audio_chunks", "bin_indices"], f"compute_audio_features has unexpected parameters: {_feat_params}"
assert not _switch_params, f"mode-switching parameters remain: {_switch_params}"
assert not _dead_branches, f"dead condition branches remain in compute_audio_features: {_dead_branches}"
print(f"[NO-MODE-SWITCH CHECK] compute_audio_features{tuple(_feat_params)} | slice_real_bimodal has no "
      f"window/compression/normalization parameters | source scan for hann/log1p/log_db/zscore/elif "
      f"branches: {'clean' if _feat_src is not None else 'source unavailable (signature check only)'} ✅")
if _stale_globals:
    print(f"⚠️  stale globals from an older notebook are still in this kernel: {_stale_globals} "
          f"(unused by v9 — restart the kernel for a clean run)")
else:
    print("[NO-MODE-SWITCH CHECK] no experiment globals (window / compression / normalization options) and no leftover N / Ta / sensor-feature globals defined ✅")

# 4. Mission-Level Stratification across all 99 Flights
class_missions = defaultdict(list)
for c_dir in sorted(DATASET_ROOT.iterdir()):
    if not c_dir.is_dir() or c_dir.name.startswith("."): continue
    try: c_label = int(c_dir.name)
    except ValueError: continue
    for m_dir in sorted(c_dir.iterdir()):
        if m_dir.is_dir() and not m_dir.name.startswith("."):
            class_missions[c_label].append(m_dir)

random.seed(42)

train_mission_map, val_mission_map, test_mission_map = {}, {}, {}
for c_label in sorted(class_missions.keys()):
    missions = class_missions[c_label]
    random.shuffle(missions)
    n_train = max(1, int(len(missions) * 0.70))
    n_val = max(1, int(len(missions) * 0.15))

    train_mission_map[c_label] = missions[:n_train]
    val_mission_map[c_label]   = missions[n_train:n_train + n_val]
    test_mission_map[c_label]  = missions[n_train + n_val:]

    print(f"Class {c_label} ({len(missions)} flights) -> "
          f"Train: {len(train_mission_map[c_label])}, "
          f"Val: {len(val_mission_map[c_label])}, "
          f"Test: {len(test_mission_map[c_label])}")

# --- 5. Strongest-64 bins: selected ONCE from TRAINING missions only, frozen ---
# The FREQUENCY axis does not depend on the audio TIME axis (Ta), so the same
# frozen indices are reused for every Ta condition and for train/val/test.
all_train_missions = [m for missions in train_mission_map.values() for m in missions]
_heldout_missions = {m for mp in (val_mission_map, test_mission_map) for ms in mp.values() for m in ms}
assert not (set(all_train_missions) & _heldout_missions), (
    "LEAKAGE: training missions overlap validation/test missions")

SELECTED_BIN_INDICES, AUDIO_BIN_WIDTH_HZ = select_strongest_audio_bins(all_train_missions, n_bins=N_AUDIO_BINS)
_bin_fingerprint_at_selection = (SELECTED_BIN_INDICES[:5].tolist(), SELECTED_BIN_INDICES[-5:].tolist())
print(f"[STRONGEST-{N_AUDIO_BINS}] frozen once from {len(all_train_missions)} TRAINING missions "
      f"(0 of {len(_heldout_missions)} validation/test missions used); reused unchanged for every Ta.")


# --- 6. Per-Ta preprocessing (called once per Ta by run_experiment in Cell 6) ---
def preprocess_for_ta(Ta):
    """Train/val/test windows with Ta audio time positions per window, using the
    frozen strongest-64 bins. Sensor side (TS_SENSOR positions, 6 channels) is
    built by exactly the same code as v6 and does not depend on Ta."""
    global _fft_shape_printed
    _fft_shape_printed = False
    _audio_chunks_per_window.clear()

    split_maps = [("TRAIN", train_mission_map), ("VAL", val_mission_map), ("TEST", test_mission_map)]
    split_samples = {}
    _fp_train = None
    for split_label, mission_map in split_maps:
        note = "(frozen strongest-64 bins)" if split_label == "TRAIN" else "(same frozen bin indices — NOT recomputed)"
        print(f"\n--- Preprocessing {split_label} split, Ta={Ta} {note} ---")
        samples = []
        for c_label in sorted(class_missions.keys()):
            for m in tqdm(mission_map[c_label], desc=f"{split_label.capitalize()} Class {c_label} (Ta={Ta})", leave=False):
                p = get_mission(m)
                if p:
                    samples.extend(slice_real_bimodal(p, c_label, SELECTED_BIN_INDICES, Ta=Ta, Ts=TS_SENSOR))
        split_samples[split_label] = samples

        _fp = (SELECTED_BIN_INDICES[:5].tolist(), SELECTED_BIN_INDICES[-5:].tolist())
        if split_label == "TRAIN":
            _fp_train = _fp
            print(f"[STRONGEST-{N_AUDIO_BINS}] Frozen bin indices applied @ TRAIN (first5, last5): {_fp} "
                  f"| matches selection: {_fp == _bin_fingerprint_at_selection}")
        else:
            print(f"[STRONGEST-{N_AUDIO_BINS}] Frozen bin indices applied @ {split_label:<5} (first5, last5): {_fp} "
                  f"| matches TRAIN: {_fp == _fp_train}")

    # --- Shape verification across all three splits:
    #     audio  (n_windows, Ta, N_AUDIO_BINS)  = (windows, TIME, FREQUENCY)
    #     sensor (n_windows, TS_SENSOR, 6)      — independent of Ta
    sensor_fingerprint = {}
    for split_label, samples in split_samples.items():
        x_a_all = np.stack([s["x_a"] for s in samples])
        tau_a_all = np.stack([s["tau_a"] for s in samples])
        x_s_all = np.stack([s["x_s"] for s in samples])
        tau_s_all = np.stack([s["tau_s"] for s in samples])
        assert x_a_all.shape == (len(samples), Ta, N_AUDIO_BINS), (
            f"Ta={Ta} FAILED: {split_label} audio array is {x_a_all.shape}, "
            f"expected (n_windows={len(samples)}, Ta={Ta}, N_AUDIO_BINS={N_AUDIO_BINS})")
        assert tau_a_all.shape == (len(samples), Ta), f"Ta={Ta} FAILED: {split_label} tau_a is {tau_a_all.shape}"
        assert x_s_all.shape == (len(samples), TS_SENSOR, 6), (
            f"Ta={Ta} FAILED: {split_label} sensor array is {x_s_all.shape}, expected (n_windows, {TS_SENSOR}, 6)")
        assert tau_s_all.shape == (len(samples), TS_SENSOR), f"Ta={Ta} FAILED: {split_label} tau_s is {tau_s_all.shape}"
        sensor_fingerprint[split_label] = (len(samples), round(float(x_s_all.astype(np.float64).sum()), 3),
                                           round(float(tau_s_all.astype(np.float64).sum()), 3),
                                           tuple(np.bincount([s["label"] for s in samples], minlength=5).tolist()))
        print(f"[SHAPE CHECK] {split_label:<5}: audio {x_a_all.shape} = (n_windows, Ta={Ta}, bins={N_AUDIO_BINS}) ✅ | "
              f"sensor {x_s_all.shape} = (n_windows, Ts={TS_SENSOR}, 6) ✅")
        del x_a_all, tau_a_all, x_s_all, tau_s_all

    cpw = np.array(_audio_chunks_per_window)
    n_short = int((cpw < Ta).sum())
    print(f"[AUDIO RESOLUTION] real audio chunks inside a {2.56:.2f}s window: min {cpw.min()} | median "
          f"{int(np.median(cpw))} | max {cpw.max()} -> Ta={Ta} "
          + ("sub-samples the real chunks in every window." if n_short == 0 else
             f"exceeds the available chunks in {n_short}/{len(cpw)} windows ({n_short / len(cpw) * 100:.1f}%), "
             "where some real chunk positions are repeated (no synthetic values are created)."))

    print("\n" + "=" * 60)
    print(f"     100% REAL PREPROCESSING COMPLETE (Ta={Ta}, ZERO SYNTHETIC DATA)   ")
    if _chunk_duration_log:
        _arr = np.array(_chunk_duration_log)
        print(f"Audio chunk duration check -> mean: {_arr.mean()*1000:.2f} ms | min: {_arr.min()*1000:.2f} ms | max: {_arr.max()*1000:.2f} ms")
    print("=" * 60)
    print(f"  • Total Train Diagnostic Windows : {len(split_samples['TRAIN'])}")
    print(f"  • Total Val Diagnostic Windows   : {len(split_samples['VAL'])}")
    print(f"  • Total Test Diagnostic Windows  : {len(split_samples['TEST'])}")
    print("=" * 60)

    out_dir = CACHE_DIR / f"Ta{Ta}"
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = {"train": out_dir / "bimodal_train.pt", "val": out_dir / "bimodal_val.pt", "test": out_dir / "bimodal_test.pt"}
    torch.save(split_samples["TRAIN"], paths["train"])
    torch.save(split_samples["VAL"], paths["val"])
    torch.save(split_samples["TEST"], paths["test"])
    print(f"✅ Cached real bimodal dataset for Ta={Ta} in {out_dir}")

    return {
        "paths": paths,
        "n_windows": (len(split_samples["TRAIN"]), len(split_samples["VAL"]), len(split_samples["TEST"])),
        "sensor_fingerprint": sensor_fingerprint,
        "chunks_per_window": (int(cpw.min()), int(np.median(cpw)), int(cpw.max())),
        "pct_windows_ta_exceeds_chunks": round(n_short / len(cpw) * 100, 2),
    }

print(f"\n✅ Preprocessing helpers ready — split + strongest-{N_AUDIO_BINS} bins fixed above; window "
      f"preprocessing (Ta = {TA_AUDIO}, Ts = {TS_SENSOR}) is built once in Cell 6 and reused for every beta.")

# Cell 5: DataLoader (weights_only=False for PyTorch 2.6+)

In [ ]:
class RealBiModalDataset(Dataset):
    def __init__(self, pt_path: Path, mean=None, std=None):
        self.samples = torch.load(pt_path, weights_only=False)
        self.Ta = int(self.samples[0]["x_a"].shape[0])  # audio TIME positions per window
        self.Ts = 128

        if mean is None or std is None:
            all_s = np.concatenate([s["x_s"] for s in self.samples], axis=0)
            self.mean = all_s.mean(axis=0, keepdims=True)
            self.std = all_s.std(axis=0, keepdims=True) + 1e-6
        else:
            self.mean = mean
            self.std = std

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        norm_sensor = (s["x_s"] - self.mean) / self.std

        # --- FIX 1: use the REAL per-window timestamps saved during
        # preprocessing (Cell 4), normalized by that window's real duration,
        # instead of a synthetic torch.linspace(0.0, 1.0, ...) that was
        # identical for every sample regardless of the underlying data.
        window_sec = float(s["window_sec"])
        tau_a = torch.tensor(s["tau_a"], dtype=torch.float32) / window_sec
        tau_s = torch.tensor(s["tau_s"], dtype=torch.float32) / window_sec

        return {
            "x_a": torch.tensor(s["x_a"], dtype=torch.float32),
            "x_s": torch.tensor(norm_sensor, dtype=torch.float32),
            "tau_a": tau_a,
            "tau_s": tau_s,
            "label": torch.tensor(s["label"], dtype=torch.long),
        }

batch_size = 32

def build_dataloaders(paths, batch_size=batch_size):
    """Datasets + loaders (same settings as v7; identical for every beta)."""
    train_ds = RealBiModalDataset(paths["train"])
    val_ds   = RealBiModalDataset(paths["val"], mean=train_ds.mean, std=train_ds.std)
    test_ds  = RealBiModalDataset(paths["test"], mean=train_ds.mean, std=train_ds.std)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=2)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=2)
    test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=2)

    print(f"DataLoaders ready -> Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)} windows")
    return train_ds, val_ds, test_ds, train_loader, val_loader, test_loader

print("RealBiModalDataset / build_dataloaders ready (DataLoaders are built per beta in Cell 6).")

In [ ]:
# ==============================================================================
# FIX 1 VERIFICATION HOOK — confirm tau_a / tau_s carry REAL, per-sample-varying
# timestamps, not the old synthetic torch.linspace(0.0, 1.0, ...) that was
# identical for every sample regardless of the underlying data.
# ==============================================================================
def verify_real_taus(train_ds):
    print("=" * 70)
    print("VERIFICATION HOOK — FIX 1: real per-sample tau_a / tau_s")
    print("=" * 70)

    check_idxs = [0, 1, len(train_ds) // 2, len(train_ds) - 1]
    taus_a, taus_s = [], []
    for idx in check_idxs:
        item = train_ds[idx]
        taus_a.append(item["tau_a"])
        taus_s.append(item["tau_s"])
        print(f"sample idx={idx:>6} | tau_a[:5]={item['tau_a'][:5].numpy()} | tau_s[:5]={item['tau_s'][:5].numpy()}")

    all_tau_a_identical = all(torch.allclose(taus_a[0], t) for t in taus_a[1:])
    all_tau_s_identical = all(torch.allclose(taus_s[0], t) for t in taus_s[1:])

    assert not all_tau_a_identical, (
        "❌ FIX 1 FAILED: tau_a is IDENTICAL across every sample checked — "
        "real timestamps are NOT reaching the model."
    )
    assert not all_tau_s_identical, (
        "❌ FIX 1 FAILED: tau_s is IDENTICAL across every sample checked — "
        "real timestamps are NOT reaching the model."
    )
    print("✅ FIX 1 VERIFIED: tau_a and tau_s vary across real samples (not synthetic/fixed).")
    print("=" * 70)

print("verify_real_taus(train_ds) ready — runs for every beta inside run_experiment.")

# Cell 6: run_condition(num_blocks, d_state, d_conv) — one isolated condition (restored mlp g(Δτ), configurable Mamba)

In [ ]:
# ==============================================================================
# run_condition(num_blocks, d_state, d_conv) — ONE complete, isolated condition:
#   seeds reset -> v10 preprocessing (Ta=64, strongest-64, raw 6 IMU; built once,
#   identical for every condition) -> DataLoaders -> FRESH model (beta=1.0, mlp
#   g(Δτ) 97 params, 4 LACA heads, this condition's Mamba config) +
#   class-weighted loss + optimizer + scheduler -> 20-epoch training with
#   best-val checkpoint -> unseen-test evaluation (+ learned lag Δ_as) -> GPU
#   memory freed.
#
# Where the Mamba configuration lives (Cell 2):
#   BiModalAeroSyncMamba(num_mamba_blocks=n, mamba_d_state=s, mamba_d_conv=c)
#     -> build_fusion_blocks -> n x RealMambaBlock -> mamba_ssm.Mamba(d_model=256,
#        d_state=s, d_conv=c, expand=2)
# v5-v10 values, read from v10: num_mamba_blocks=4 (passed in run_experiment),
# d_conv=4 (hard-coded in the block constructor), d_state=16 (RealMambaBlock's
# default, never overridden), expand=2.
# ==============================================================================
import gc
import hashlib

EPOCHS = 20
D_MODEL = 256
_PREP = None   # v10 preprocessing result (Ta=64), built once and reused by every condition

# Fingerprints printed by the earlier v10 runs (Ran_minor-project-v10.ipynb "[INIT]" line): sha1 over every
# initial state_dict tensor EXCEPT the lag-bias MLP, and of the torch CPU RNG state after construction.
# The restored v12 baseline builds the model exactly as v9/v10 did, so both should match bit-for-bit.
V10_INIT_FP_PREFIX = "6430fb5f8eb2"
V10_RNG_FP_PREFIX = "5641b401720d"


def _sha1_state(model, exclude):
    h = hashlib.sha1()
    for k, v in model.state_dict().items():
        if any(e in k for e in exclude):
            continue
        h.update(k.encode())
        h.update(v.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()


def init_fingerprints(model):
    """non-lag-bias init (comparable with v10), non-Mamba init (must be identical across every Mamba
    config), and the torch CPU RNG state right after construction."""
    return {
        "non_lag_bias": _sha1_state(model, (".lag_bias.",)),
        "non_mamba": _sha1_state(model, ("mamba_blocks.",)),
        "rng": hashlib.sha1(torch.get_rng_state().numpy().tobytes()).hexdigest(),
    }


def lag_bias_param_arithmetic(g):
    """(exact learnable-parameter count, hand-checkable arithmetic) for the Linear/Tanh/Linear g(Δτ)."""
    terms = []
    for m in g.modules():
        if isinstance(m, nn.Linear):
            terms.append(f"Linear({m.in_features},{m.out_features}) weights {m.weight.numel()}")
            terms.append(f"bias {m.bias.numel()}")
    total = sum(p.numel() for p in g.parameters())
    return total, " + ".join(terms) + f" = {total} params"


def verify_restored_lag_bias(model):
    """Part 1: the mlp g(Δτ) is back to the original v5-v9 Linear(1,32)-Tanh-Linear(32,1)."""
    for name in ("laca_as", "laca_sa"):
        laca = getattr(model.laca, name)
        g = laca.lag_bias
        n, arith = lag_bias_param_arithmetic(g)
        print(f"[LAG-BIAS] model.laca.{name}.lag_bias: {arith} | num_heads={laca.num_heads} | beta={laca.beta}")
        assert (isinstance(g, nn.Sequential) and len(g) == 3 and isinstance(g[1], nn.Tanh)
                and (g[0].in_features, g[0].out_features, g[2].in_features, g[2].out_features) == (1, 32, 32, 1)), (
            f"{name}.lag_bias is not the original mlp: {g}")
        assert n == 97, f"{name}.lag_bias has {n} params, expected 97"
        assert laca.num_heads == 4 and laca.beta == LACA_BETA
    print("[LAG-BIAS] ✅ restored to the original mlp: 1*32 + 32 + 32*1 + 1 = 97 params (both LACA directions)")


def mamba_param_counts(model):
    per_block = [sum(p.numel() for p in b.parameters()) for b in model.mamba_blocks]
    return sum(per_block), per_block


def verify_mamba_config(model, num_blocks, d_state, d_conv, force_pytorch_conv):
    """Confirm, from the CONSTRUCTED blocks (not the config), which Mamba configuration is in use."""
    blocks = model.mamba_blocks
    assert len(blocks) == num_blocks, f"{len(blocks)} Mamba blocks built, requested {num_blocks}"
    expected_impl = resolve_conv_impl(d_conv, force_pytorch_conv)
    for i, b in enumerate(blocks):
        if isinstance(b, RealMambaBlock):
            m = b.mamba
            assert (m.d_state, m.d_conv, m.expand, m.d_model) == (d_state, d_conv, 2, D_MODEL), (
                f"block {i}: d_state={m.d_state} d_conv={m.d_conv} expand={m.expand} d_model={m.d_model}")
            assert m.conv1d.kernel_size == (d_conv,), f"block {i}: conv kernel {m.conv1d.kernel_size}"
            assert tuple(m.A_log.shape) == (m.d_inner, d_state), f"block {i}: A_log {tuple(m.A_log.shape)}"
            assert b.conv_impl == expected_impl, f"block {i}: conv_impl {b.conv_impl} != {expected_impl}"
        else:
            assert b.conv1d.kernel_size == (d_conv,), f"fallback block {i}: conv kernel {b.conv1d.kernel_size}"
    b0 = blocks[0]
    total, per_block = mamba_param_counts(model)
    if isinstance(b0, RealMambaBlock):
        m = b0.mamba
        print(f"[MAMBA CHECK] {len(blocks)} x RealMambaBlock | d_state={m.d_state} (A_log {tuple(m.A_log.shape)}) | "
              f"d_conv={m.d_conv} (conv1d kernel {m.conv1d.kernel_size[0]}) | expand={m.expand} | d_inner={m.d_inner} | "
              f"local-conv implementation: {b0.conv_impl}")
    else:
        print(f"[MAMBA CHECK] {len(blocks)} x VectorizedSSMBlock (FALLBACK — no SSM state; d_state does not apply) | "
              f"d_conv={b0.conv1d.kernel_size[0]}")
    print(f"[MAMBA PARAMS] fusion backbone total = {total:,} ({len(per_block)} blocks x {per_block[0]:,}) | "
          f"whole model = {sum(p.numel() for p in model.parameters()):,}")
    print(f"[MAMBA CHECK] ✅ constructed backbone matches num_blocks={num_blocks}, d_state={d_state}, d_conv={d_conv}")
    return total, per_block[0], b0.conv_impl if isinstance(b0, RealMambaBlock) else "native"


def verify_shapes(model, batch_t, when):
    """Real batch through the model's OWN modules; assert every shape the fusion step relies on. For fused
    blocks, also compare the fused and PyTorch conv paths on block 0 (same math, different kernel).
    Eval mode + no_grad: no RNG consumed, no BatchNorm stats updated."""
    xa, xs, ta, ts = batch_t
    was_training = model.training
    model.eval()
    with torch.no_grad():
        za = model.audio_encoder(xa)
        zs = model.sensor_encoder(xs.transpose(1, 2)).transpose(1, 2)
        Ha = model.htt(za, ta, modality_idx=0)
        Hs = model.htt(zs, ts, modality_idx=1)
        B, Ta_, Ts_ = Ha.shape[0], Ha.shape[1], Hs.shape[1]
        H_tilde_a, H_tilde_s, aux = model.laca(Ha, Hs, ta, ts)
        assert H_tilde_a.shape == (B, Ta_, D_MODEL) and H_tilde_s.shape == (B, Ts_, D_MODEL)
        F_seq = torch.cat([H_tilde_a, model.sep_token.expand(B, -1, -1), H_tilde_s], dim=1)
        L = Ta_ + 1 + Ts_
        assert F_seq.shape == (B, L, D_MODEL), f"Mamba input {tuple(F_seq.shape)}"
        blk0_in = F_seq
        for i, blk in enumerate(model.mamba_blocks):
            F_seq = blk(F_seq)
            assert F_seq.shape == (B, L, D_MODEL) and torch.isfinite(F_seq).all(), (
                f"Mamba block {i} output {tuple(F_seq.shape)} (finite={bool(torch.isfinite(F_seq).all())})")
        logits, pooled, aux_preds, aux2 = model(xa, xs, ta, ts)
        assert logits.shape == (B, model.n_classes) and torch.isfinite(logits).all(), f"logits {tuple(logits.shape)}"
        assert pooled["z_a"].shape == pooled["z_s"].shape == (B, D_MODEL)
        assert aux2["lag_as"].shape == aux2["lag_sa"].shape == (B,)
        msg = (f"[SHAPES {when}] Mamba input {tuple(blk0_in.shape)} -> {len(model.mamba_blocks)} blocks, each "
               f"{(B, L, D_MODEL)} ✅ -> logits {tuple(logits.shape)} ✅")
        b0 = model.mamba_blocks[0]
        if isinstance(b0, RealMambaBlock) and b0.conv_impl == "fused":
            fused = b0(blk0_in)
            with _causal_conv1d_kernel_disabled():
                b0.mamba.use_fast_path = False
                try:
                    unfused = b0.norm(blk0_in)
                    unfused = blk0_in + b0.mamba(unfused)
                finally:
                    b0.mamba.use_fast_path = True
            msg += f" | block 0 fused vs PyTorch-conv path max|diff| = {float((fused - unfused).abs().max()):.2e}"
        print(msg)
    model.train(was_training)


def run_condition(num_blocks, d_state, d_conv, force_pytorch_conv=False, tag=None, seed=SEED, epochs=EPOCHS):
    global _PREP
    t_run0 = time.time()
    tag = tag or f"blocks={num_blocks},d_state={d_state},d_conv={d_conv}"
    set_all_seeds(seed)
    print(f"[SEEDS] reset at condition start -> random / numpy / torch / cuda = {seed}")
    print(f"[CONDITION] {tag} | requested Mamba config: num_blocks={num_blocks}, d_state={d_state}, d_conv={d_conv}"
          f"{' (PyTorch conv path FORCED)' if force_pytorch_conv else ''} | fixed: beta={LACA_BETA}, mlp g(Δτ), 4 heads")

    # --- 1. v10 preprocessing (Ta=64) — identical for every condition, built on first use ---
    if _PREP is None:
        _PREP = preprocess_for_ta(TA_AUDIO)
    else:
        print(f"[DATA] reusing the Ta={TA_AUDIO} windows built for the first condition (identical data for every condition)")
    prep = _PREP

    # --- 2. Datasets / loaders (v10 settings) ---
    train_ds, val_ds, test_ds, train_loader, val_loader, test_loader = build_dataloaders(prep["paths"])
    verify_real_taus(train_ds)

    # Batch built straight from the dataset (no DataLoader iteration -> no RNG consumed).
    probe = [train_ds[i] for i in range(min(batch_size, len(train_ds)))]
    probe_t = tuple(torch.stack([p[k] for p in probe]).to(device) for k in ("x_a", "x_s", "tau_a", "tau_s"))
    assert tuple(probe_t[0].shape) == (len(probe), TA_AUDIO, N_AUDIO_BINS)
    assert tuple(probe_t[1].shape) == (len(probe), TS_SENSOR, 6)

    # --- 3. FRESH model with this condition's Mamba config (seed reset right before, exactly as v5-v10) ---
    set_all_seeds(seed)
    model = BiModalAeroSyncMamba(
        d_model=D_MODEL,
        n_classes=5,
        num_mamba_blocks=num_blocks,          # Stage A variable
        dropout=0.15,
        temporal_bias_scale=LACA_BETA,        # FIXED at 1.0
        audio_in_dim=N_AUDIO_BINS,
        sensor_in_dim=6,
        mamba_d_state=d_state,                # Stage B variable
        mamba_d_conv=d_conv,                  # Stage C variable
        mamba_force_pytorch_conv=force_pytorch_conv,
    ).to(device)
    fps = init_fingerprints(model)

    print("\n" + "-" * 90)
    verify_restored_lag_bias(model)
    mamba_total, mamba_per_block, conv_impl = verify_mamba_config(model, num_blocks, d_state, d_conv, force_pytorch_conv)
    same_v10 = fps["non_lag_bias"].startswith(V10_INIT_FP_PREFIX) and fps["rng"].startswith(V10_RNG_FP_PREFIX)
    if (num_blocks, d_state, d_conv) == DEFAULT_MAMBA_CONFIG:
        print(f"[INIT] non-lag-bias init {fps['non_lag_bias'][:12]} / RNG {fps['rng'][:12]} vs v10 "
              f"({V10_INIT_FP_PREFIX} / {V10_RNG_FP_PREFIX}): "
              + ("IDENTICAL ✅ (model starts exactly as in v9/v10)" if same_v10 else
                 "DIFFERENT ⚠️  (init or RNG stream differs from v9/v10; a different torch build can also cause this)"))
    ref = _BASELINE_FPS if "_BASELINE_FPS" in globals() else None
    if ref is not None:
        same_nm, same_rng = fps["non_mamba"] == ref["non_mamba"], fps["rng"] == ref["rng"]
        print(f"[INIT] every NON-Mamba initial weight identical to the Part 1 baseline: {'YES ✅' if same_nm else 'NO ❌'}"
              f" | RNG state after construction identical (-> same shuffle order + dropout masks): "
              f"{'YES ✅' if same_rng else 'NO ❌'}")
        if not (same_nm and same_rng):
            print("🚨" * 40 + "\n🚨  the rest of the network / RNG stream differs from the baseline — this condition "
                  "does NOT isolate the Mamba configuration.\n" + "🚨" * 40)
    else:
        same_nm = same_rng = None
    verify_shapes(model, probe_t, "before training")
    print("-" * 90 + "\n")

    _fusion_cls = type(model.mamba_blocks[0])
    print(f"[MODEL] FUSION BACKBONE IN USE: {_fusion_cls.__module__}.{_fusion_cls.__name__}"
          + ("  🚨 pure-PyTorch FALLBACK, NOT real mamba_ssm 🚨" if _fusion_cls.__name__ == "VectorizedSSMBlock"
             else "  ✅ REAL mamba_ssm selective-scan SSM"))

    # FIX 4: inverse-frequency class weights from REAL training-set class counts
    train_labels_arr = np.array([s["label"] for s in train_ds.samples])
    class_counts = np.bincount(train_labels_arr, minlength=5)
    class_weights_np = class_counts.sum() / (len(class_counts) * np.maximum(class_counts, 1))
    class_weights = torch.tensor(class_weights_np, dtype=torch.float32).to(device)
    print("[FIX 4] class weights (from real train counts): "
          + " | ".join(f"C{c}: n={class_counts[c]} w={class_weights_np[c]:.4f}" for c in range(5)))

    criterion = BiModalAeroSyncLoss(lambda_align=0.1, lambda_cons=0.1, class_weights=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    best_val_f1 = 0.0
    best_weights = None
    best_epoch = None
    history = {"train_loss": [], "val_loss": [], "val_f1": [], "val_acc": []}
    ckpt_path = CACHE_DIR / (f"aerosync_bimodal_best_mamba_b{num_blocks}_s{d_state}_c{d_conv}"
                             f"{'_ptconv' if force_pytorch_conv else ''}.pt")

    try:
        # --- 4. Training (identical loop to v5-v10) ---
        start_time = time.time()
        print(f"🚀 Training Bi-Modal AeroSync-Mamba ({tag}) on {len(train_ds)} real flight windows...\n")

        for epoch in range(1, epochs + 1):
            model.train()
            running_loss, running_cls = 0.0, 0.0

            pbar = tqdm(train_loader, desc=f"{tag} Epoch {epoch:02d}/{epochs}", leave=False)
            for batch in pbar:
                xa = batch["x_a"].to(device, non_blocking=True)
                xs = batch["x_s"].to(device, non_blocking=True)
                ta = batch["tau_a"].to(device, non_blocking=True)
                ts = batch["tau_s"].to(device, non_blocking=True)
                labels = batch["label"].to(device, non_blocking=True)

                optimizer.zero_grad()
                logits, pooled_reps, aux_preds, _ = model(xa, xs, ta, ts)
                loss, l_dict = criterion(logits, labels, pooled_reps, aux_preds)
                loss.backward()

                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

                running_loss += loss.item()
                running_cls += l_dict["loss_cls"]
                pbar.set_postfix({"loss": f"{loss.item():.3f}", "cls": f"{l_dict['loss_cls']:.3f}"})

            scheduler.step()
            avg_train_loss = running_loss / len(train_loader)

            # --- Validation ---
            model.eval()
            val_preds, val_targets, val_probs = [], [], []
            val_loss_total = 0.0

            with torch.no_grad():
                for batch in val_loader:
                    xa = batch["x_a"].to(device)
                    xs = batch["x_s"].to(device)
                    ta = batch["tau_a"].to(device)
                    ts = batch["tau_s"].to(device)
                    labels = batch["label"].to(device)

                    logits, pooled_reps, aux_preds, _ = model(xa, xs, ta, ts)
                    v_loss, _ = criterion(logits, labels, pooled_reps, aux_preds)
                    val_loss_total += v_loss.item()

                    probs = F.softmax(logits, dim=-1)
                    val_preds.extend(probs.argmax(dim=-1).cpu().numpy())
                    val_targets.extend(labels.cpu().numpy())
                    val_probs.extend(probs.cpu().numpy())

            avg_val_loss = val_loss_total / len(val_loader)
            val_metrics = compute_all_metrics(np.array(val_targets), np.array(val_preds), np.array(val_probs))

            history["train_loss"].append(avg_train_loss)
            history["val_loss"].append(avg_val_loss)
            history["val_f1"].append(val_metrics["macro_f1"])
            history["val_acc"].append(val_metrics["accuracy"])

            is_best = ""
            if val_metrics["macro_f1"] > best_val_f1:
                best_val_f1 = val_metrics["macro_f1"]
                best_weights = copy.deepcopy(model.state_dict())
                best_epoch = epoch
                torch.save(best_weights, ckpt_path)
                is_best = " ⭐ [Best Saved]"

            print(
                f"{tag} Epoch {epoch:02d}/{epochs} | "
                f"Train Loss: {avg_train_loss:.4f} | "
                f"Val Loss: {avg_val_loss:.4f} | "
                f"Val Acc: {val_metrics['accuracy']:.4f} | "
                f"Val Macro-F1: {val_metrics['macro_f1']:.4f}{is_best}"
            )

        train_seconds = time.time() - start_time
        print(f"\n🎉 {tag} Training Finished in {train_seconds:.1f}s | Best Val Macro-F1: {best_val_f1:.4f} "
              f"(epoch {best_epoch})")

        # --- 5. Unseen test evaluation with the best-val weights (as v5 Cell 7, incl. learned lag) ---
        if best_weights is not None:
            model.load_state_dict(best_weights)
        model.eval()
        verify_shapes(model, probe_t, "best checkpoint")

        test_preds, test_targets, test_probs, lags_as_list, lags_sa_list = [], [], [], [], []
        with torch.no_grad():
            for batch in test_loader:
                xa = batch["x_a"].to(device)
                xs = batch["x_s"].to(device)
                ta = batch["tau_a"].to(device)
                ts = batch["tau_s"].to(device)
                labels = batch["label"].to(device)

                logits, _, _, aux = model(xa, xs, ta, ts)
                probs = F.softmax(logits, dim=-1)
                test_preds.extend(probs.argmax(dim=-1).cpu().numpy())
                test_targets.extend(labels.cpu().numpy())
                test_probs.extend(probs.cpu().numpy())
                lags_as_list.extend(aux["lag_as"].cpu().numpy())
                lags_sa_list.extend(aux["lag_sa"].cpu().numpy())

        test_metrics = compute_all_metrics(np.array(test_targets), np.array(test_preds), np.array(test_probs))
        lag_as_mean = float(np.mean(lags_as_list))
        lag_as_std = float(np.std(lags_as_list))
        lag_sa_mean = float(np.mean(lags_sa_list))
        n_test = len(test_targets)
        print("=" * 65)
        print(f"     BI-MODAL AEROSYNC-MAMBA ({tag}): 100% REAL TEST BENCHMARK")
        print("=" * 65)
        print(f"  • Test Accuracy       : {test_metrics['accuracy']:.4f} ({test_metrics['accuracy']*100:.2f}%)")
        print(f"  • Macro-F1 Score      : {test_metrics['macro_f1']:.4f}")
        print(f"  • Balanced Accuracy   : {test_metrics['balanced_accuracy']:.4f}")
        print(f"  • One-vs-Rest AUROC   : {test_metrics['macro_auroc']:.4f}")
        print(f"  • Learned Audio-IMU Lag (Δ_as) : {lag_as_mean:+.4f} (normalized units) "
              f"≈ {lag_as_mean * WINDOW_SEC * 1000:+.1f} ms | std over test windows {lag_as_std:.4f}")
        print(f"  • Learned IMU-Audio Lag (Δ_sa) : {lag_sa_mean:+.4f} (normalized units)")
        print(f"  • Mamba backbone params : {mamba_total:,} ({num_blocks} blocks, d_state={d_state}, d_conv={d_conv}, "
              f"conv={conv_impl})")
        print("=" * 65)
    finally:
        # --- 6. Free GPU memory so sequential conditions never accumulate ---
        del model, optimizer, scheduler, criterion, best_weights, probe_t, probe
        del train_loader, val_loader, test_loader, train_ds, val_ds, test_ds
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            print(f"[GPU] freed | allocated={torch.cuda.memory_allocated() / 1024**2:.1f} MB "
                  f"| reserved={torch.cuda.memory_reserved() / 1024**2:.1f} MB")

    return {
        "num_mamba_blocks": num_blocks,
        "d_state": d_state,
        "d_conv": d_conv,
        "conv_impl": conv_impl,
        "accuracy": float(test_metrics["accuracy"]),
        "macro_f1": float(test_metrics["macro_f1"]),
        "balanced_accuracy": float(test_metrics["balanced_accuracy"]),
        "macro_auroc": float(test_metrics["macro_auroc"]),
        "learned_lag_as_mean": round(lag_as_mean, 6),
        "train_seconds": round(train_seconds, 1),
        "mamba_params": mamba_total,
        "mamba_params_per_block": mamba_per_block,
        "learned_lag_as_std": round(lag_as_std, 6),
        "learned_lag_as_ms": round(lag_as_mean * WINDOW_SEC * 1000, 2),
        "learned_lag_sa_mean": round(lag_sa_mean, 6),
        "init_matches_v10": same_v10 if (num_blocks, d_state, d_conv) == DEFAULT_MAMBA_CONFIG else None,
        "non_mamba_init_matches_baseline": same_nm,
        "rng_matches_baseline": same_rng,
        "best_val_macro_f1": round(float(best_val_f1), 6),
        "best_epoch": best_epoch,
        "n_test_windows": n_test,
        "seed": seed,
        "run_seconds": round(time.time() - t_run0, 1),
        # baseline reproduction fingerprints (not written to the CSV)
        "_fps": fps,
        "_n_windows": prep["n_windows"],
        "_class_counts": class_counts.tolist(),
        "_sensor_fingerprint": prep["sensor_fingerprint"],
        "_epoch1": {"train_loss": history["train_loss"][0], "val_loss": history["val_loss"][0],
                    "val_acc": history["val_acc"][0], "val_f1": history["val_f1"][0]},
    }

WINDOW_SEC = 2.56   # slice_real_bimodal window length; tau is normalized by it in RealBiModalDataset
print(f"run_condition(num_blocks, d_state, d_conv) ready | default Mamba config {DEFAULT_MAMBA_CONFIG} | "
      f"fused causal-conv1d kernel available: {_fused_conv_available()} (widths {CAUSAL_CONV1D_WIDTHS}) | "
      f"fixed: beta={LACA_BETA}, mlp g(Δτ) 97 params, 4 LACA heads, Ta={TA_AUDIO}, N={N_AUDIO_BINS}, Ts={TS_SENSOR}")

# Cell 7: PART 1 — restored baseline must reproduce v10 (acc 0.9009, macro-F1 0.8966, bal-acc 0.8940, AUROC 0.9884) before Part 2

In [ ]:
# ==============================================================================
# PART 1 — restored baseline (mlp g(Δτ) back to 97 params; Mamba 4 blocks,
# d_state=16, d_conv=4) must reproduce v9/v10 BEFORE Part 2 runs. This run is
# also Stage A's num_blocks=4 condition (reused there, not re-trained).
# Results CSV + reuse bookkeeping for the whole notebook live here too.
# ==============================================================================
import csv
import traceback
from datetime import datetime

SWEEP_CSV = Path("/kaggle/working/mamba_config_sweep_results.csv")
SWEEP_CSV_FIELDS = ["stage", "condition_id", "num_mamba_blocks", "d_state", "d_conv", "conv_impl",
                    "accuracy", "macro_f1", "balanced_accuracy", "macro_auroc", "learned_lag_as_mean",
                    "train_seconds", "mamba_params", "mamba_params_per_block", "status",
                    "non_mamba_init_matches_baseline", "rng_matches_baseline", "init_matches_v10",
                    "learned_lag_as_std", "learned_lag_as_ms", "learned_lag_sa_mean", "best_val_macro_f1",
                    "best_epoch", "n_test_windows", "seed", "run_seconds", "note", "finished_at"]

# If True, Part 2 does NOT run when the Part 1 baseline fails to reproduce v9/v10.
REQUIRE_BASELINE_REPRODUCTION = True

# Known result for the restored configuration — verified v5-v10 (and v11 num_heads=4).
V10_BASELINE = {"accuracy": 0.9009, "macro_f1": 0.8966, "balanced_accuracy": 0.8940, "macro_auroc": 0.9884}
V5_LAG_AS = -0.0196
# Deterministic fingerprints of the v7-v10 data path — independent of GPU numerics, must match EXACTLY.
V10_BIN_INDICES_N64 = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255,
                       258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 501, 502, 503, 504, 505, 506, 507, 508,
                       509, 510, 511, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 754, 757, 758, 759, 760,
                       761, 762, 763, 764, 765, 766]
V10_N_WINDOWS = (4934, 954, 1130)
V10_CLASS_COUNTS = [906, 1020, 1004, 1008, 996]
V10_SENSOR_FINGERPRINT = {
    "TRAIN": (4934, -6167105.225, 808427.456, (906, 1020, 1004, 1008, 996)),
    "VAL":   (954, -1193206.652, 156301.664, (126, 229, 199, 190, 210)),
    "TEST":  (1130, -1413563.944, 185127.438, (286, 205, 210, 228, 201)),
}
V7_EPOCH1 = {"train_loss": 1.3863, "val_loss": 0.8762, "val_acc": 0.8249, "val_f1": 0.8390}
REPRO_TOL = 0.0005

_RESULTS_BY_CONFIG = {}   # (num_blocks, d_state, d_conv, force_pytorch_conv) -> finished row, for reuse


def append_row(row):
    out = {k: row.get(k) for k in SWEEP_CSV_FIELDS}
    out["finished_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    write_header = not SWEEP_CSV.exists()
    with open(SWEEP_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=SWEEP_CSV_FIELDS)
        if write_header:
            w.writeheader()
        w.writerow(out)
    print(f"💾 appended {row['stage']} / {row['condition_id']} to {SWEEP_CSV} ({'header + row' if write_header else 'row'})")


def run_or_reuse(stage, condition_id, num_blocks, d_state, d_conv, force_pytorch_conv=False, note=""):
    """Either reuse an identical already-run configuration (same rule as the v4 audio 3-stage sweep) or
    train it. The row is appended to the CSV right away either way."""
    key = (num_blocks, d_state, d_conv, force_pytorch_conv)
    print(f"[CONFIG] {stage}/{condition_id}: num_blocks={num_blocks} | d_state={d_state} | d_conv={d_conv}"
          f"{' | PyTorch conv path FORCED' if force_pytorch_conv else ''}")
    if key in _RESULTS_BY_CONFIG:
        prev = _RESULTS_BY_CONFIG[key]
        print(f"♻️  REUSED, NOT RE-RUN: {stage}/{condition_id} has a configuration identical to already-computed "
              f"{prev['stage']}/{prev['condition_id']} -> copying its metrics (macro_f1={prev['macro_f1']:.4f}, "
              f"Mamba params {prev['mamba_params']:,}).")
        row = {k: v for k, v in prev.items() if not k.startswith("_")}
        row.update({"stage": stage, "condition_id": condition_id, "train_seconds": 0.0, "run_seconds": 0.0,
                    "status": f"reused from {prev['stage']}/{prev['condition_id']} (not re-run)", "note": note})
        append_row(row)
        return row
    res = run_condition(num_blocks, d_state, d_conv, force_pytorch_conv,
                        tag=f"{stage}/{condition_id} b={num_blocks},s={d_state},c={d_conv}"
                            + (",ptconv" if force_pytorch_conv else ""))
    res.update({"stage": stage, "condition_id": condition_id, "status": "computed", "note": note})
    _RESULTS_BY_CONFIG[key] = res
    append_row(res)
    return res


def check_baseline_against_v10(res):
    """Loud reproduction check for the restored baseline. Returns (ok, one-line verdict)."""
    print("\n" + "=" * 90)
    print("  PART 1 CHECK — does the restored baseline (mlp 97 params, 4 x Mamba d_state=16 d_conv=4) reproduce v9/v10?")
    print("=" * 90)
    det = {
        "strongest-64 bin indices": SELECTED_BIN_INDICES.tolist() == V10_BIN_INDICES_N64,
        f"train/val/test windows {V10_N_WINDOWS}": tuple(res["_n_windows"]) == V10_N_WINDOWS,
        f"train class counts {V10_CLASS_COUNTS}": res["_class_counts"] == V10_CLASS_COUNTS,
        "raw sensor values/timestamps/labels (v10 fingerprint)": res["_sensor_fingerprint"] == V10_SENSOR_FINGERPRINT,
    }
    for name, ok in det.items():
        print(f"  [deterministic] {name:<53}: {'MATCH ✅' if ok else 'MISMATCH ❌'}")
    print(f"  [init]          {'model init + RNG vs v10 (' + V10_INIT_FP_PREFIX + ')':<53}: "
          f"{'MATCH ✅' if res['init_matches_v10'] else 'MISMATCH ⚠️'}")
    e1 = res["_epoch1"]
    print("  [epoch 1]       " + " | ".join(
        f"{k}: v7 {V7_EPOCH1[k]:.4f} vs now {e1[k]:.4f} ({e1[k] - V7_EPOCH1[k]:+.4f})" for k in V7_EPOCH1))

    print(f"\n  {'metric':<20} {'v10 known':>10} {'this run':>10} {'delta':>10}")
    worst = 0.0
    for k, v in V10_BASELINE.items():
        d = res[k] - v
        worst = max(worst, abs(d))
        print(f"  {k:<20} {v:>10.4f} {res[k]:>10.4f} {d:>+10.4f}{'' if abs(d) <= REPRO_TOL else '   <-- differs'}")
    print(f"  {'learned lag Δ_as':<20} {V5_LAG_AS:>+10.4f} {res['learned_lag_as_mean']:>+10.4f} "
          f"{res['learned_lag_as_mean'] - V5_LAG_AS:>+10.4f}   (v5 Cell 7 value; informational)")

    if not all(det.values()):
        ok, verdict = False, "PORT BUG: deterministic v10 fingerprint mismatch"
        print("\n" + "🚨" * 40)
        print("🚨  PART 1 FAILED — bins / windows / class counts / sensor values differ from v10, which cannot be")
        print("🚨  GPU noise: the data path changed in the v10 -> v12 port.")
        print("🚨" * 40)
    elif worst <= REPRO_TOL:
        ok, verdict = True, "reproduces v10 (all 4 metrics within 0.0005)"
        print(f"\n✅ PART 1 PASSED: the restored baseline REPRODUCES v10 — all four test metrics within ±{REPRO_TOL} and all "
              f"deterministic fingerprints match.")
    else:
        ok, verdict = False, f"DOES NOT reproduce v10 metrics (max |delta| {worst:.4f}); data fingerprints match"
        print("\n" + "🚨" * 40)
        print(f"🚨  PART 1 FAILED — TEST METRICS DIFFER FROM v10 (largest |delta| = {worst:.4f} > {REPRO_TOL}).")
        print("🚨  Data path is identical, so the difference arose in training. Check the [init] and epoch-1 lines:")
        print("🚨  if init or epoch 1 already differ, the model/RNG path changed in the port; if they match and later")
        print("🚨  epochs drift, it is GPU non-determinism.")
        print("🚨" * 40)
    print("=" * 90)
    return ok, verdict


if SWEEP_CSV.exists():
    _backup = SWEEP_CSV.with_suffix(f".{datetime.now().strftime('%Y%m%d_%H%M%S')}.bak.csv")
    SWEEP_CSV.rename(_backup)
    print(f"ℹ️  existing results file moved to {_backup}")

_sweep_t0 = time.time()
BASELINE_ROW, BASELINE_OK, BASELINE_VERDICT = None, False, "not run"
print("\n\n" + "=" * 90)
print(f"=== PART 1: restored baseline — mlp g(Δτ) 97 params, Mamba {DEFAULT_MAMBA_CONFIG} (num_blocks, d_state, d_conv) ===")
print("=" * 90)
try:
    set_all_seeds(SEED)
    BASELINE_ROW = run_condition(*DEFAULT_MAMBA_CONFIG, tag="P1/baseline b=4,s=16,c=4")
    BASELINE_ROW.update({"stage": "P1", "condition_id": "baseline", "status": "computed"})
    _BASELINE_FPS = BASELINE_ROW["_fps"]
    _RESULTS_BY_CONFIG[DEFAULT_MAMBA_CONFIG + (False,)] = BASELINE_ROW
    BASELINE_OK, BASELINE_VERDICT = check_baseline_against_v10(BASELINE_ROW)
    BASELINE_ROW["note"] = BASELINE_VERDICT
    append_row(BASELINE_ROW)
except Exception as exc:
    BASELINE_VERDICT = f"baseline run FAILED: {exc!r}"
    print(f"\n❌ PART 1 baseline FAILED: {exc!r}")
    traceback.print_exc()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

PART2_ENABLED = BASELINE_OK or not REQUIRE_BASELINE_REPRODUCTION
if not BASELINE_OK:
    print("\n" + "🚨" * 40)
    print(f"🚨  PART 1 did not pass ({BASELINE_VERDICT}).")
    print("🚨  " + ("Part 2 will NOT run (REQUIRE_BASELINE_REPRODUCTION=True). Fix the reproduction first, or set it to "
                   "False to run the sweep anyway." if REQUIRE_BASELINE_REPRODUCTION else
                   "REQUIRE_BASELINE_REPRODUCTION=False -> Part 2 runs anyway; treat its results with suspicion."))
    print("🚨" * 40)
else:
    print(f"\n✅ Part 1 verified — proceeding to Part 2 (Mamba configuration sweep). "
          f"Elapsed {(time.time() - _sweep_t0) / 60:.1f} min.")

# Cell 8: PART 2 — Mamba configuration sweep: Stage A num_blocks -> Stage B d_state -> Stage C d_conv (winner locked per stage, noise-floor rule)

In [ ]:
# ==============================================================================
# PART 2 — Mamba configuration sweep, SEQUENTIAL (not a grid), one variable per
# stage, each stage's winner locked in before the next:
#   Stage A  num_blocks ∈ {2, 4, 6}   (d_state=16, d_conv=4)
#   Stage B  d_state    ∈ {8, 16, 32} (num_blocks = A winner, d_conv=4)
#   Stage C  d_conv     ∈ {2, 4, 6}   (num_blocks, d_state = A + B winners)
# The default of each stage is the previous stage's winner configuration, so it
# is REUSED (not re-trained), exactly like the v4 audio 3-stage sweep.
#
# Winner rule (noise-floor discipline): the highest test macro-F1 wins ONLY if it
# beats the stage's default by MORE than ±NOISE_BAND_F1; otherwise the default
# is retained and the nominal top is flagged as within noise. Exact ties go to
# the default. Every thin margin is flagged explicitly.
#
# d_conv=6: the fused causal-conv1d CUDA kernel only supports widths 2-4, so
# d_conv=6 runs on mamba_ssm's PyTorch nn.Conv1d path (see Cell 2). Stage C then
# also trains a CONTROL — d_conv=4 forced onto that same PyTorch path — so that
# d_conv=6 is judged against d_conv=4 on the SAME implementation, and the
# implementation's own effect on the metrics is visible. The control is not a
# candidate (it has the same configuration as the default).
# ==============================================================================
NOISE_BAND_F1 = 0.0005   # spread between near-identical same-seed conditions in earlier sweeps
STAGE_A_BLOCKS = [2, 4, 6]
STAGE_B_D_STATE = [8, 16, 32]
STAGE_C_D_CONV = [2, 4, 6]
_DEF_BLOCKS, _DEF_D_STATE, _DEF_D_CONV = DEFAULT_MAMBA_CONFIG

STAGE_WINNERS = {}
STAGE_ROWS = {}
sweep_failed = []


def _same_impl_reference(cand, default_row, control_row):
    """The row a candidate's margin is measured against: the default, or — for a PyTorch-conv candidate
    when the default ran on the fused kernel — the same-implementation control."""
    if (control_row is not None and cand.get("conv_impl") == "pytorch-forced"
            and default_row.get("conv_impl") != "pytorch-forced"):
        return control_row
    return default_row


def pick_stage_winner(stage, var, rows, default_value, control_row=None):
    rows = [r for r in rows if r is not None]
    default_row = next((r for r in rows if r[var] == default_value), None)
    ranked = sorted(rows, key=lambda r: (-r["macro_f1"], r is not default_row))   # exact ties -> default first
    top = ranked[0]
    print("\n" + "#" * 100)
    print(f"#  STAGE {stage} RESULT — {var} (noise floor ±{NOISE_BAND_F1} macro-F1)")
    print("#" * 100)
    for r in ranked:
        ref = _same_impl_reference(r, default_row, control_row) if default_row is not None else None
        d = "" if ref is None or r is default_row else (
            f" | Δ vs {'same-impl control' if ref is control_row else 'default'} {r['macro_f1'] - ref['macro_f1']:+.4f}")
        print(f"#  {var}={r[var]!s:<3} macro-F1 {r['macro_f1']:.4f} | acc {r['accuracy']:.4f} | Mamba params "
              f"{int(r['mamba_params']):>9,} | conv {r['conv_impl']:<14} | {r['status']}{d}"
              + ("   <- default" if r is default_row else ""))
    if control_row is not None and default_row is not None:
        print(f"#  CONTROL d_conv={control_row['d_conv']} on the PyTorch conv path: macro-F1 {control_row['macro_f1']:.4f} "
              f"(implementation effect vs fused default: {control_row['macro_f1'] - default_row['macro_f1']:+.4f}"
              + (" — LARGER than the noise floor: the conv implementation alone moves the result"
                 if abs(control_row['macro_f1'] - default_row['macro_f1']) > NOISE_BAND_F1 else " — within noise") + ")")

    flags = []
    if default_row is None:
        winner = top
        flags.append("default condition missing — nominal top taken without a noise-floor comparison")
    elif top is default_row:
        winner = default_row
        second = ranked[1] if len(ranked) > 1 else None
        if second is not None and top["macro_f1"] - second["macro_f1"] <= NOISE_BAND_F1:
            flags.append(f"default is only nominally best: {var}={second[var]} is within the noise floor "
                         f"({second['macro_f1'] - top['macro_f1']:+.4f})")
    else:
        ref = _same_impl_reference(top, default_row, control_row)
        margin = top["macro_f1"] - ref["macro_f1"]
        if margin > NOISE_BAND_F1:
            winner = top
            second = ranked[1]
            if top["macro_f1"] - second["macro_f1"] <= NOISE_BAND_F1:
                flags.append(f"winner's lead over the runner-up {var}={second[var]} is within the noise floor "
                             f"({top['macro_f1'] - second['macro_f1']:+.4f})")
            if margin <= 2 * NOISE_BAND_F1:
                flags.append(f"margin over the {'same-impl control' if ref is control_row else 'default'} "
                             f"({margin:+.4f}) is less than 2x the noise floor — thin")
        else:
            winner = default_row
            flags.append(f"nominal top {var}={top[var]} beats the {'same-impl control' if ref is control_row else 'default'} "
                         f"by only {margin:+.4f} (≤ ±{NOISE_BAND_F1}) -> NOT a real winner; default {var}={default_value} "
                         f"RETAINED")
    print("#")
    print(f"#  🏁 STAGE {stage} WINNER: {var} = {winner[var]}  (macro-F1 {winner['macro_f1']:.4f}, "
          f"Mamba params {int(winner['mamba_params']):,})")
    for f in flags:
        print(f"#  ⚠️  THIN / NOISE FLAG: {f}")
    if not flags:
        print(f"#  ✅ winner decided by more than the ±{NOISE_BAND_F1} noise floor")
    print("#" * 100)
    STAGE_WINNERS[stage] = {"var": var, "value": winner[var], "row": winner, "flags": flags,
                            "default_value": default_value}
    return winner


def run_stage(stage, var, candidates, fixed, default_value, extra_control=None):
    rows, control_row = [], None
    for i, value in enumerate(candidates, start=1):
        cfg = dict(fixed, **{var: value})
        print("\n\n" + "=" * 90)
        print(f"=== Stage {stage} ({var}) condition {i}/{len(candidates)}: {var}={value}"
              + (" (DEFAULT — sanity check vs baseline)" if value == default_value else "")
              + f" | fixed {', '.join(f'{k}={v}' for k, v in fixed.items())} | elapsed "
              f"{(time.time() - _sweep_t0) / 60:.1f} min ===")
        print("=" * 90)
        try:
            rows.append(run_or_reuse(stage, f"{stage}{i}", cfg["num_mamba_blocks"], cfg["d_state"], cfg["d_conv"],
                                     note="default" if value == default_value else ""))
        except Exception as exc:
            sweep_failed.append((f"{stage}{i}", repr(exc)))
            print(f"\n❌ {stage}{i} ({var}={value}) FAILED: {exc!r}")
            traceback.print_exc()
            print("↪️  continuing; this condition is excluded from the winner selection.")
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    if extra_control is not None:
        cfg = dict(fixed, **{var: extra_control})
        print("\n\n" + "=" * 90)
        print(f"=== Stage {stage} CONTROL: {var}={extra_control} on the PyTorch conv path (same implementation as the "
              f"out-of-kernel-range candidate) ===")
        print("=" * 90)
        try:
            control_row = run_or_reuse(stage, f"{stage}-ctrl", cfg["num_mamba_blocks"], cfg["d_state"], cfg["d_conv"],
                                       force_pytorch_conv=True, note="control (not a candidate)")
        except Exception as exc:
            sweep_failed.append((f"{stage}-ctrl", repr(exc)))
            print(f"\n❌ {stage}-ctrl FAILED: {exc!r}")
            traceback.print_exc()
    STAGE_ROWS[stage] = rows + ([control_row] if control_row is not None else [])
    if not rows:
        raise RuntimeError(f"Stage {stage}: every condition failed — cannot pick a winner")
    return pick_stage_winner(stage, var, rows, default_value, control_row)


if not PART2_ENABLED:
    print("⛔ PART 2 SKIPPED — Part 1 did not reproduce the v10 baseline (see the banner above). "
          "Set REQUIRE_BASELINE_REPRODUCTION=False in the Part 1 cell to run it anyway.")
else:
    # --- Pre-check: what each stage will actually run ---
    fallback = _MAMBA_CLASS is None
    print("PART 2 PLAN:")
    print(f"  Stage A num_blocks {STAGE_A_BLOCKS} | Stage B d_state {STAGE_B_D_STATE} | Stage C d_conv {STAGE_C_D_CONV}")
    for c in STAGE_C_D_CONV:
        print(f"  d_conv={c}: local-conv implementation -> {resolve_conv_impl(c)}")
    needs_control = any(resolve_conv_impl(c) == "pytorch-forced" for c in STAGE_C_D_CONV)
    if needs_control:
        print(f"  ⚠️  d_conv outside {CAUSAL_CONV1D_WIDTHS} cannot use the fused causal-conv1d kernel -> runs on the "
              f"PyTorch conv path; Stage C adds a d_conv={_DEF_D_CONV} control on that same path.")
    if fallback:
        print("  🚨 FALLBACK backbone (no mamba_ssm): VectorizedSSMBlock has no SSM state, so Stage B (d_state) "
              "is SKIPPED — its conditions would be identical models.")

    # --- STAGE A: number of Mamba blocks ---
    win_A = run_stage("A", "num_mamba_blocks", STAGE_A_BLOCKS,
                      fixed={"d_state": _DEF_D_STATE, "d_conv": _DEF_D_CONV}, default_value=_DEF_BLOCKS)
    A_BLOCKS = int(win_A["num_mamba_blocks"])

    # --- STAGE B: SSM state dimension (default = Stage A winner's exact config -> reused) ---
    if fallback:
        B_D_STATE = _DEF_D_STATE
        print(f"\n⏭️  Stage B skipped (fallback backbone) — d_state stays {B_D_STATE} (not applicable).")
    else:
        win_B = run_stage("B", "d_state", STAGE_B_D_STATE,
                          fixed={"num_mamba_blocks": A_BLOCKS, "d_conv": _DEF_D_CONV}, default_value=_DEF_D_STATE)
        B_D_STATE = int(win_B["d_state"])

    # --- STAGE C: local conv width (default = Stage A+B winners' exact config -> reused) ---
    win_C = run_stage("C", "d_conv", STAGE_C_D_CONV,
                      fixed={"num_mamba_blocks": A_BLOCKS, "d_state": B_D_STATE}, default_value=_DEF_D_CONV,
                      extra_control=_DEF_D_CONV if needs_control else None)
    C_D_CONV = int(win_C["d_conv"])

    FINAL_MAMBA_CONFIG = (A_BLOCKS, B_D_STATE, C_D_CONV)
    print("\n\n" + "=" * 90)
    print(f"PART 2 COMPLETE in {(time.time() - _sweep_t0) / 60:.1f} min (incl. Part 1) | results: {SWEEP_CSV}")
    print(f"FINAL MAMBA CONFIG: num_blocks={A_BLOCKS}, d_state={B_D_STATE}, d_conv={C_D_CONV} "
          f"(default was {DEFAULT_MAMBA_CONFIG})")
    for fid, err in sweep_failed:
        print(f"  ❌ {fid}: {err}")
    print("=" * 90)

# Cell 9: Final Results — Part 1 verdict, full table, configurations by macro-F1 with Mamba params, stage winners, final configuration

In [ ]:
# ==============================================================================
# FINAL RESULTS — reloaded from the CSV on disk (crash-safe source of truth):
# Part 1 verdict, the full table (every row, stage order), the unique trained
# configurations sorted by macro-F1 with Mamba parameter counts, each stage's
# winner + noise flags, and the final winning configuration.
# ==============================================================================
if not SWEEP_CSV.exists():
    raise FileNotFoundError(f"{SWEEP_CSV} does not exist — no condition finished. Re-run the Part 1 / Part 2 cells.")

sweep_df = pd.read_csv(SWEEP_CSV)
pd.set_option("display.width", 260)
try:
    from IPython.display import display
except Exception:
    display = lambda t: print(t.to_string(index=False))

show_cols = ["stage", "condition_id", "num_mamba_blocks", "d_state", "d_conv", "conv_impl", "accuracy", "macro_f1",
             "balanced_accuracy", "macro_auroc", "learned_lag_as_mean", "mamba_params", "train_seconds", "status", "note"]
fmt = sweep_df[show_cols].copy()
for col in ("accuracy", "macro_f1", "balanced_accuracy", "macro_auroc", "learned_lag_as_mean"):
    fmt[col] = fmt[col].round(4)

# --- 1. Part 1 ---
print("=" * 120)
p1 = sweep_df[sweep_df["stage"] == "P1"]
if len(p1):
    r = p1.iloc[0]
    print("PART 1 — restored baseline (mlp g(Δτ) 97 params, Mamba 4 x d_state 16 x d_conv 4) vs known v10:")
    for k, v in V10_BASELINE.items():
        d = r[k] - v
        print(f"  {k:<20} v10 {v:.4f} | this run {r[k]:.4f} | delta {d:+.4f}" + ("" if abs(d) <= REPRO_TOL else "   <-- differs"))
    print(f"  verdict: {r['note']}")
    if not str(r["note"]).startswith("reproduces"):
        print("🚨" * 40 + "\n🚨  PART 1 did NOT reproduce v10 — Part 2 results (if any) are not anchored.\n" + "🚨" * 40)
else:
    print("🚨 PART 1 row missing from the CSV — the baseline did not finish.")

# --- 2. Full table, stage order (every row, including reused ones) ---
print("\n" + "=" * 120)
print("   FULL RESULTS — every condition in stage order (♻️ reused rows copy an identical earlier configuration)")
print("=" * 120)
display(fmt)

# --- 3. Unique trained configurations, sorted by macro-F1 ---
trained = sweep_df[sweep_df["status"] == "computed"].sort_values("macro_f1", ascending=False, kind="stable")
trained = trained.reset_index(drop=True)
base_f1 = float(p1.iloc[0]["macro_f1"]) if len(p1) else float("nan")
uniq = trained[["num_mamba_blocks", "d_state", "d_conv", "conv_impl", "accuracy", "macro_f1", "balanced_accuracy",
                "macro_auroc", "mamba_params", "train_seconds", "stage", "condition_id"]].copy()
uniq.insert(0, "rank", np.arange(1, len(uniq) + 1))
uniq["Δ macro_f1 vs baseline"] = (uniq["macro_f1"] - base_f1).round(4)
uniq["Δ Mamba params vs baseline"] = (uniq["mamba_params"] - int(p1.iloc[0]["mamba_params"])) if len(p1) else np.nan
for col in ("accuracy", "macro_f1", "balanced_accuracy", "macro_auroc"):
    uniq[col] = uniq[col].round(4)
print("\n" + "=" * 120)
print(f"   UNIQUE TRAINED CONFIGURATIONS — sorted by test macro-F1 (noise floor ±{NOISE_BAND_F1}); Mamba capacity shown "
      f"alongside so a capacity effect is visible, not conflated")
print("=" * 120)
display(uniq)

# --- 4. Stage winners ---
print("\n" + "#" * 120)
if "STAGE_WINNERS" in globals() and STAGE_WINNERS:
    for stage in ("A", "B", "C"):
        if stage not in STAGE_WINNERS:
            print(f"#  Stage {stage}: not run")
            continue
        w = STAGE_WINNERS[stage]
        print(f"#  Stage {stage} ({w['var']}): winner {w['value']} (default {w['default_value']}) | "
              f"macro-F1 {w['row']['macro_f1']:.4f} | Mamba params {int(w['row']['mamba_params']):,}")
        for f in w["flags"]:
            print(f"#      ⚠️  {f}")
    if "FINAL_MAMBA_CONFIG" in globals():
        fb, fs, fc = FINAL_MAMBA_CONFIG
        fr = STAGE_WINNERS["C"]["row"]   # Stage C winner = the final configuration's own result row
        print("#")
        print(f"#  🏆 FINAL WINNING MAMBA CONFIGURATION: num_blocks={fb}, d_state={fs}, d_conv={fc}"
              + ("  (= the v5-v10 default — no change)" if FINAL_MAMBA_CONFIG == DEFAULT_MAMBA_CONFIG else ""))
        print(f"#     test acc {fr['accuracy']:.4f} | macro-F1 {fr['macro_f1']:.4f} ({fr['macro_f1'] - base_f1:+.4f} vs "
              f"baseline) | bal-acc {fr['balanced_accuracy']:.4f} | AUROC {fr['macro_auroc']:.4f} | Mamba params "
              f"{int(fr['mamba_params']):,} | conv {fr['conv_impl']}")
        n_flags = sum(len(STAGE_WINNERS[s]["flags"]) for s in STAGE_WINNERS)
        if n_flags:
            print(f"#     ⚠️  {n_flags} thin-margin / noise flag(s) across the stages (listed above) — scrutinize before adopting.")
else:
    print("#  Part 2 did not run (see the Part 1 verdict above).")
print("#" * 120)

print("\n   Caveat: same-seed reruns here are bit-exact, so the noise floor is the spread between near-identical "
      "conditions, not a measured seed-to-seed variance; one seed per condition cannot establish a small effect.")
print(f"📄 full results CSV: {SWEEP_CSV}")